# CDPL UCI HAR / PAMAP2 Training Notebook


## Imports and dependencies

Import all Python, NumPy/Pandas, scikit-learn, tqdm, and PyTorch dependencies used by the training and evaluation pipeline.

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import shutil
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Union

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

## Reproducibility

Defines the random seed and PyTorch backend settings used for repeatable runs.

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

## Configuration

Central configuration dataclass for dataset paths, model dimensions, optimization, personalization, stabilization, and method selection.

In [ ]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    # ----- data -----
    # dataset_name can be "pamap2" or "uci_har".
    # For UCI HAR, set data_dir to the folder that contains:
    #   train/Inertial Signals, train/y_train.txt, train/subject_train.txt,
    #   test/Inertial Signals,  test/y_test.txt,  test/subject_test.txt
    dataset_name: str = "uci_har"
    data_dir: str = "/content/drive/MyDrive/UCI HAR Dataset"
    subject_glob: str = "subject*.dat"
    valid_activity_ids: Tuple[int, ...] = (1, 2, 3, 4, 5, 6, 7, 12, 13, 16, 17, 24)

    # UCI HAR README: windows are already 2.56 s at 50 Hz = 128 readings, with 50% overlap, and 9 raw inertial channels.
    # PAMAP2 uses raw rows and creates windows using seq_len/stride.
    seq_len: int = 128
    stride: int = 64
    drop_mixed_windows: bool = True
    purge_gap_raw: int = 50  # PAMAP2 only: gap between raw sub-splits
    uci_purge_gap_windows: int = 1  # UCI only: gap between pre-windowed sub-splits

    train_subject_train_ratio: float = 0.85
    train_subject_val_ratio: float = 0.15

    heldout_support_ratio: float = 0.20
    heldout_adapt_val_ratio: float = 0.20
    heldout_test_ratio: float = 0.60

    min_windows_per_partition: int = 8

    # ----- optimization -----
    batch_size: int = 128
    personal_batch_size: int = 64
    num_workers: int = 0
    pin_memory: bool = True

    rounds: int = 15
    local_epochs: int = 6
    personalization_epochs: int = 10

    lr_encoder: float = 3e-4
    lr_personal: float = 1e-2
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    lambda_align: float = 0.7
    lambda_A: float = 1e-3
    lambda_B: float = 5e-3

    server_geom_steps: int = 20
    server_geom_lr: float = 2e-2
    server_alt_iters: int = 3

    # ----- model -----
    d_model: int = 192
    emb_dim: int = 192
    n_heads: int = 6
    n_layers: int = 3
    ff_dim: int = 384
    dropout: float = 0.2
    proto_rank: int = 8

    # ----- misc -----
    ece_bins: int = 15
    seed: int = 42
    amp: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "./cdpl_uci_har_runs"

    # ----- stabilization / scoring -----
    class_weight_power: float = 0.5
    class_weight_max: float = 4.0
    min_proto_samples_per_class: int = 4

    tau_init: float = 8.0
    tau_min: float = 1.5
    tau_max: float = 20.0
    lambda_tau: float = 1e-4   # reduced from 5e-4

    lambda_personal_anchor: float = 3e-4   # reduced from 2e-3

    server_proto_momentum: float = 0.85
    server_basis_momentum: float = 0.80

    client_val_weight_power: float = 0.5
    round_score_f1_w: float = 0.70
    round_score_acc_w: float = 0.30
    round_score_ece_w: float = 0.05

    # ----- adaptive personalization -----
    personal_tau_lr_mult: float = 0.25
    personalization_epochs_min: int = 6
    personalization_epochs_max: int = 14
    personalization_patience: int = 3
    hard_subject_f1_threshold: float = 0.55
    easy_subject_f1_threshold: float = 0.78

    # ----- personalization gating -----
    personalization_gate_score_margin: float = 0.002
    personalization_gate_score_margin_easy: float = 0.010
    personalization_gate_f1_margin: float = 0.000

    # for faster debugging
    run_single_heldout: Optional[str] = None  # UCI HAR example: "subject01"

    # Review-2 multi-method UCI HAR benchmark.
    # Valid entries: "fedavg", "fedproto", "fedrep", "cdpl".
    methods_to_run: Tuple[str, ...] = ("fedavg", "fedproto", "fedrep", "cdpl")

    # Colab/Drive safety: copy UCI HAR text files to /content before np.loadtxt.
    cache_drive_dataset_locally: bool = True
    local_data_cache_dir: str = "/content/uci_har_cached"

    def __post_init__(self) -> None:
        assert abs(self.train_subject_train_ratio + self.train_subject_val_ratio - 1.0) < 1e-8
        assert abs(
            self.heldout_support_ratio + self.heldout_adapt_val_ratio + self.heldout_test_ratio - 1.0
        ) < 1e-8
        os.makedirs(self.save_dir, exist_ok=True)

## PAMAP2 columns

Column definitions and feature selection utilities for PAMAP2-style sensor files.

In [ ]:
# ============================================================
# PAMAP2 columns
# ============================================================

def pamap2_columns() -> List[str]:
    return [
        "timestamp",
        "activity_id",
        "heart_rate",
        "hand_temperature",
        "hand_acc16_x",
        "hand_acc16_y",
        "hand_acc16_z",
        "hand_acc6_x",
        "hand_acc6_y",
        "hand_acc6_z",
        "hand_gyro_x",
        "hand_gyro_y",
        "hand_gyro_z",
        "hand_mag_x",
        "hand_mag_y",
        "hand_mag_z",
        "hand_orient_1",
        "hand_orient_2",
        "hand_orient_3",
        "hand_orient_4",
        "chest_temperature",
        "chest_acc16_x",
        "chest_acc16_y",
        "chest_acc16_z",
        "chest_acc6_x",
        "chest_acc6_y",
        "chest_acc6_z",
        "chest_gyro_x",
        "chest_gyro_y",
        "chest_gyro_z",
        "chest_mag_x",
        "chest_mag_y",
        "chest_mag_z",
        "chest_orient_1",
        "chest_orient_2",
        "chest_orient_3",
        "chest_orient_4",
        "ankle_temperature",
        "ankle_acc16_x",
        "ankle_acc16_y",
        "ankle_acc16_z",
        "ankle_acc6_x",
        "ankle_acc6_y",
        "ankle_acc6_z",
        "ankle_gyro_x",
        "ankle_gyro_y",
        "ankle_gyro_z",
        "ankle_mag_x",
        "ankle_mag_y",
        "ankle_mag_z",
        "ankle_orient_1",
        "ankle_orient_2",
        "ankle_orient_3",
        "ankle_orient_4",
    ]


def feature_columns() -> List[str]:
    cols = pamap2_columns()
    return [c for c in cols if c not in ("timestamp", "activity_id")]

## Metrics

Accuracy, Macro-F1, Expected Calibration Error, Brier score, weighted aggregation, and personalization selection scoring utilities.

In [ ]:
# ============================================================
# Metrics
# ============================================================

def expected_calibration_error(
    probs: np.ndarray,
    labels: np.ndarray,
    n_bins: int = 15,
) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)
        if mask.any():
            acc_bin = accuracies[mask].mean()
            conf_bin = confidences[mask].mean()
            ece += float(mask.mean()) * abs(float(acc_bin) - float(conf_bin))
    return float(ece)


def multiclass_brier_score(probs: np.ndarray, labels: np.ndarray, num_classes: int) -> float:
    one_hot = np.eye(num_classes, dtype=np.float32)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def summarize_probs(
    probs: np.ndarray,
    labels: np.ndarray,
    num_classes: int,
    ece_bins: int,
) -> Dict[str, float]:
    preds = probs.argmax(axis=1)
    return {
        "acc": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "ece": expected_calibration_error(probs, labels, n_bins=ece_bins),
        "brier": multiclass_brier_score(probs, labels, num_classes=num_classes),
    }

def weighted_mean(values: List[float], weights: List[float]) -> float:
    if len(values) == 0:
        return 0.0
    w = np.asarray(weights, dtype=np.float64)
    v = np.asarray(values, dtype=np.float64)
    w = w / max(w.sum(), 1e-12)
    return float(np.sum(w * v))


def aggregate_round_metrics(
    per_client_metrics: Dict[str, Dict[str, float]],
    per_client_val_sizes: Dict[str, int],
    cfg: Config,
) -> Tuple[float, float, float, float]:
    client_ids = sorted(per_client_metrics.keys())
    weights = [max(1, per_client_val_sizes[sid]) ** cfg.client_val_weight_power for sid in client_ids]

    mean_f1 = weighted_mean([per_client_metrics[sid]["f1"] for sid in client_ids], weights)
    mean_acc = weighted_mean([per_client_metrics[sid]["acc"] for sid in client_ids], weights)
    mean_ece = weighted_mean([per_client_metrics[sid]["ece"] for sid in client_ids], weights)

    score = (
        cfg.round_score_f1_w * mean_f1
        + cfg.round_score_acc_w * mean_acc
        - cfg.round_score_ece_w * mean_ece
    )
    return mean_f1, mean_acc, mean_ece, score

def choose_personalization_hparams(
    init_f1: float,
    cfg: Config,
) -> Dict[str, float]:
    """
    Hard subjects get more epochs and weaker anchoring.
    Easy subjects get fewer epochs and slightly stronger anchoring.
    """
    if init_f1 < cfg.hard_subject_f1_threshold:
        return {
            "epochs": cfg.personalization_epochs_max,
            "anchor_w": cfg.lambda_personal_anchor * 0.35,
            "lr_A": cfg.lr_personal,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult * 0.75,
            "warm_epochs": 2,
        }
    elif init_f1 > cfg.easy_subject_f1_threshold:
        return {
            "epochs": cfg.personalization_epochs_min,
            "anchor_w": cfg.lambda_personal_anchor * 2.0,
            "lr_A": cfg.lr_personal * 0.85,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult * 0.50,
            "warm_epochs": 1,
        }
    else:
        return {
            "epochs": int(round(0.5 * (cfg.personalization_epochs_min + cfg.personalization_epochs_max))),
            "anchor_w": cfg.lambda_personal_anchor,
            "lr_A": cfg.lr_personal,
            "lr_tau": cfg.lr_personal * cfg.personal_tau_lr_mult,
            "warm_epochs": 2,
        }

def personalization_selection_score(metrics: Dict[str, float]) -> float:
    return 0.90 * metrics["f1"] + 0.10 * metrics["acc"] - 0.05 * metrics["ece"]

## Data structures

Dataset, subject partition, fold data, and full-training dataclasses used throughout the pipeline.

In [ ]:
# ============================================================
# Data structures
# ============================================================

class WindowDataset(Dataset):
    def __init__(self, windows: np.ndarray, labels: np.ndarray):
        assert len(windows) == len(labels)
        self.x = torch.from_numpy(np.ascontiguousarray(windows)).float()
        self.y = torch.from_numpy(np.ascontiguousarray(labels)).long()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


@dataclass
class SubjectPartition:
    windows: np.ndarray
    labels: np.ndarray


@dataclass
class FoldData:
    heldout_id: str
    train_ids: List[str]
    train_subjects: Dict[str, Dict[str, SubjectPartition]]
    heldout_subject: Dict[str, SubjectPartition]
    num_classes: int
    feature_dim: int
    label_map: Dict[int, int]
    scaler: StandardScaler


@dataclass
class FullTrainingData:
    subject_ids: List[str]
    train_subjects: Dict[str, Dict[str, SubjectPartition]]
    num_classes: int
    feature_dim: int
    label_map: Dict[int, int]
    scaler: StandardScaler

## Loading and preprocessing

PAMAP2 loading, interpolation, label mapping, leakage-safe window generation, subject partitioning, scaling, and inference-bundle helpers.

In [ ]:
# ============================================================
# Loading and preprocessing
# ============================================================

def discover_subject_files(cfg: Config) -> Dict[str, Path]:
    paths = sorted(Path(cfg.data_dir).glob(cfg.subject_glob))
    if not paths:
        raise FileNotFoundError(
            f"No PAMAP2 files found in {cfg.data_dir!r} with glob {cfg.subject_glob!r}."
        )
    return {p.stem: p for p in paths}


def load_subject_dataframe(file_path: Path, cfg: Config) -> pd.DataFrame:
    cols = pamap2_columns()
    df = pd.read_csv(file_path, sep=r"\s+", header=None, names=cols, engine="python")

    # only keep valid activities
    df = df[df["activity_id"].isin(cfg.valid_activity_ids)].copy()
    if df.empty:
        raise RuntimeError(f"{file_path.name} has no rows for valid PAMAP2 activity IDs.")

    # PAMAP2 uses -1 as missing marker
    df.replace(-1.0, np.nan, inplace=True)

    feat_cols = feature_columns()
    df[feat_cols] = df[feat_cols].interpolate(method="linear", limit_direction="both", axis=0)
    df[feat_cols] = df[feat_cols].ffill().bfill()
    df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median())

    return df.reset_index(drop=True)


def build_global_label_map(subject_dfs: Dict[str, pd.DataFrame]) -> Dict[int, int]:
    activity_ids = sorted({int(a) for df in subject_dfs.values() for a in df["activity_id"].unique()})
    return {aid: idx for idx, aid in enumerate(activity_ids)}


def encode_subject(
    df: pd.DataFrame,
    label_map: Dict[int, int],
) -> Tuple[np.ndarray, np.ndarray]:
    X = df[feature_columns()].to_numpy(dtype=np.float32)
    y = df["activity_id"].map(label_map).to_numpy(dtype=np.int64)
    return X, y


def constant_label_segments(y: np.ndarray) -> List[Tuple[int, int, int]]:
    segments: List[Tuple[int, int, int]] = []
    if len(y) == 0:
        return segments

    s = 0
    n = len(y)
    while s < n:
        e = s + 1
        while e < n and y[e] == y[s]:
            e += 1
        segments.append((s, e, int(y[s])))
        s = e
    return segments


def generate_constant_label_windows(
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
    stride: int,
    drop_mixed_windows: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    windows: List[np.ndarray] = []
    labels: List[int] = []

    if len(X) == 0:
        feat_dim = X.shape[1] if X.ndim == 2 else 0
        return np.empty((0, seq_len, feat_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)

    s = 0
    n = len(y)
    while s < n:
        e = s + 1
        while e < n and y[e] == y[s]:
            e += 1

        seg_x = X[s:e]
        seg_y = y[s]
        seg_len = len(seg_x)

        if seg_len >= seq_len:
            for start in range(0, seg_len - seq_len + 1, stride):
                win_x = seg_x[start : start + seq_len]
                if drop_mixed_windows and not np.all(y[s + start : s + start + seq_len] == seg_y):
                    continue
                windows.append(win_x)
                labels.append(int(seg_y))
        s = e

    feat_dim = X.shape[1]
    if not windows:
        return np.empty((0, seq_len, feat_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.stack(windows).astype(np.float32), np.array(labels, dtype=np.int64)


def empty_partition(seq_len: int, feat_dim: int) -> SubjectPartition:
    return SubjectPartition(
        windows=np.empty((0, seq_len, feat_dim), dtype=np.float32),
        labels=np.empty((0,), dtype=np.int64),
    )


def concat_partitions(parts: List[SubjectPartition], seq_len: int, feat_dim: int) -> SubjectPartition:
    non_empty = [p for p in parts if len(p.labels) > 0]
    if not non_empty:
        return empty_partition(seq_len, feat_dim)
    return SubjectPartition(
        windows=np.concatenate([p.windows for p in non_empty], axis=0).astype(np.float32),
        labels=np.concatenate([p.labels for p in non_empty], axis=0).astype(np.int64),
    )


def labels_present(labels: np.ndarray) -> List[int]:
    if len(labels) == 0:
        return []
    return np.where(np.bincount(labels) > 0)[0].tolist()


def label_histogram(labels: np.ndarray, num_classes: int) -> Dict[int, int]:
    if len(labels) == 0:
        return {}
    counts = np.bincount(labels, minlength=num_classes)
    return {i: int(c) for i, c in enumerate(counts) if c > 0}


def allocate_lengths_with_minimum(
    total_len: int,
    ratios: List[float],
    min_lengths: List[int],
) -> Optional[List[int]]:
    """
    Split total_len into k parts:
    - each part >= min_lengths[i]
    - remaining budget allocated by ratios
    """
    min_sum = int(sum(min_lengths))
    if total_len < min_sum:
        return None

    ratios_arr = np.array(ratios, dtype=np.float64)
    if ratios_arr.sum() <= 0:
        ratios_arr = np.ones_like(ratios_arr)
    ratios_arr = ratios_arr / ratios_arr.sum()

    extra_total = int(total_len - min_sum)
    raw_extra = ratios_arr * extra_total
    extra = np.floor(raw_extra).astype(np.int64)

    remainder = extra_total - int(extra.sum())
    if remainder > 0:
        frac = raw_extra - extra
        order = np.argsort(-frac)
        for idx in order[:remainder]:
            extra[idx] += 1

    out = (np.array(min_lengths, dtype=np.int64) + extra).astype(np.int64)
    return [int(v) for v in out]


def split_single_segment_train_val(
    seg_len: int,
    cfg: Config,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Split ONE constant-label segment into train/val subranges with a purge gap,
    so train and val both contain the class whenever the segment is long enough.
    """
    gap = cfg.purge_gap_raw
    min_win = cfg.seq_len

    # Need room for train + gap + val
    if seg_len < (2 * min_win + gap):
        return {
            "train": [(0, seg_len)] if seg_len >= min_win else [],
            "val": [],
        }

    usable = seg_len - gap
    lengths = allocate_lengths_with_minimum(
        total_len=usable,
        ratios=[cfg.train_subject_train_ratio, cfg.train_subject_val_ratio],
        min_lengths=[min_win, min_win],
    )
    if lengths is None:
        return {
            "train": [(0, seg_len)] if seg_len >= min_win else [],
            "val": [],
        }

    train_len, val_len = lengths
    train_range = (0, train_len)
    val_range = (train_len + gap, train_len + gap + val_len)

    return {
        "train": [train_range],
        "val": [val_range],
    }


def split_single_segment_support_adapt_test(
    seg_len: int,
    cfg: Config,
) -> Dict[str, List[Tuple[int, int]]]:
    """
    Split ONE constant-label segment into support/adapt_val/test subranges with purge gaps,
    so all three held-out partitions contain the class whenever the segment is long enough.
    """
    gap = cfg.purge_gap_raw
    min_win = cfg.seq_len

    # Need room for support + gap + adapt + gap + test
    if seg_len < (3 * min_win + 2 * gap):
        # fallback hierarchy: if not enough for 3-way, try 2-way, else put all in test
        if seg_len >= (2 * min_win + gap):
            usable = seg_len - gap
            lengths = allocate_lengths_with_minimum(
                total_len=usable,
                ratios=[cfg.heldout_support_ratio + cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
                min_lengths=[min_win, min_win],
            )
            if lengths is None:
                return {
                    "support": [],
                    "adapt_val": [],
                    "test": [(0, seg_len)] if seg_len >= min_win else [],
                }

            sa_len, test_len = lengths

            # Try splitting the first block into support + adapt_val too
            if sa_len >= (2 * min_win + gap):
                sa_usable = sa_len - gap
                sa_lengths = allocate_lengths_with_minimum(
                    total_len=sa_usable,
                    ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio],
                    min_lengths=[min_win, min_win],
                )
                if sa_lengths is not None:
                    support_len, adapt_len = sa_lengths
                    return {
                        "support": [(0, support_len)],
                        "adapt_val": [(support_len + gap, support_len + gap + adapt_len)],
                        "test": [(sa_len + gap, sa_len + gap + test_len)],
                    }

            # If still impossible, use support + test and leave adapt empty
            return {
                "support": [(0, sa_len)],
                "adapt_val": [],
                "test": [(sa_len + gap, sa_len + gap + test_len)],
            }

        return {
            "support": [],
            "adapt_val": [],
            "test": [(0, seg_len)] if seg_len >= min_win else [],
        }

    usable = seg_len - 2 * gap
    lengths = allocate_lengths_with_minimum(
        total_len=usable,
        ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
        min_lengths=[min_win, min_win, min_win],
    )
    if lengths is None:
        return {
            "support": [],
            "adapt_val": [],
            "test": [(0, seg_len)] if seg_len >= min_win else [],
        }

    support_len, adapt_len, test_len = lengths

    support_range = (0, support_len)
    adapt_range = (support_len + gap, support_len + gap + adapt_len)
    test_range = (support_len + gap + adapt_len + gap, support_len + gap + adapt_len + gap + test_len)

    return {
        "support": [support_range],
        "adapt_val": [adapt_range],
        "test": [test_range],
    }


def windows_from_absolute_ranges(
    X: np.ndarray,
    y: np.ndarray,
    abs_ranges: List[Tuple[int, int]],
    cfg: Config,
) -> SubjectPartition:
    feat_dim = X.shape[1]
    parts: List[SubjectPartition] = []

    for a, b in abs_ranges:
        if (b - a) < cfg.seq_len:
            continue
        w, lab = generate_constant_label_windows(
            X[a:b],
            y[a:b],
            seq_len=cfg.seq_len,
            stride=cfg.stride,
            drop_mixed_windows=cfg.drop_mixed_windows,
        )
        if len(lab) > 0:
            parts.append(SubjectPartition(windows=w, labels=lab))

    return concat_partitions(parts, cfg.seq_len, feat_dim)


def build_subject_partitions_train(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    feat_dim = X.shape[1]
    train_parts: List[SubjectPartition] = []
    val_parts: List[SubjectPartition] = []

    segments = constant_label_segments(y)

    for s, e, _lab in segments:
        seg_len = e - s
        rel = split_single_segment_train_val(seg_len, cfg)

        train_abs = [(s + a, s + b) for a, b in rel["train"]]
        val_abs = [(s + a, s + b) for a, b in rel["val"]]

        train_parts.append(windows_from_absolute_ranges(X, y, train_abs, cfg))
        val_parts.append(windows_from_absolute_ranges(X, y, val_abs, cfg))

    return {
        "train": concat_partitions(train_parts, cfg.seq_len, feat_dim),
        "val": concat_partitions(val_parts, cfg.seq_len, feat_dim),
    }


def build_subject_partitions_heldout(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    feat_dim = X.shape[1]
    support_parts: List[SubjectPartition] = []
    adapt_parts: List[SubjectPartition] = []
    test_parts: List[SubjectPartition] = []

    segments = constant_label_segments(y)

    for s, e, _lab in segments:
        seg_len = e - s
        rel = split_single_segment_support_adapt_test(seg_len, cfg)

        support_abs = [(s + a, s + b) for a, b in rel["support"]]
        adapt_abs = [(s + a, s + b) for a, b in rel["adapt_val"]]
        test_abs = [(s + a, s + b) for a, b in rel["test"]]

        support_parts.append(windows_from_absolute_ranges(X, y, support_abs, cfg))
        adapt_parts.append(windows_from_absolute_ranges(X, y, adapt_abs, cfg))
        test_parts.append(windows_from_absolute_ranges(X, y, test_abs, cfg))

    return {
        "support": concat_partitions(support_parts, cfg.seq_len, feat_dim),
        "adapt_val": concat_partitions(adapt_parts, cfg.seq_len, feat_dim),
        "test": concat_partitions(test_parts, cfg.seq_len, feat_dim),
    }


def print_partition_debug(
    sid: str,
    packaged: Dict[str, SubjectPartition],
    num_classes: int,
    heldout: bool,
) -> None:
    if heldout:
        split_order = ["support", "adapt_val", "test"]
    else:
        split_order = ["train", "val"]

    print(f"[PARTITIONS][{sid}]")
    for split_name in split_order:
        part = packaged[split_name]
        print(
            f"  {split_name}: windows={len(part.labels)}, "
            f"classes={labels_present(part.labels)}, "
            f"hist={label_histogram(part.labels, num_classes)}"
        )

    if not heldout:
        train_set = set(labels_present(packaged["train"].labels))
        val_set = set(labels_present(packaged["val"].labels))
        missing = sorted(val_set - train_set)
        print(f"  train/val overlap ok? missing_val_in_train={missing}")
    else:
        support_set = set(labels_present(packaged["support"].labels))
        adapt_set = set(labels_present(packaged["adapt_val"].labels))
        test_set = set(labels_present(packaged["test"].labels))
        print(f"  support∩adapt classes={sorted(support_set & adapt_set)}")
        print(f"  support∩test classes={sorted(support_set & test_set)}")
        print(f"  adapt∩test classes={sorted(adapt_set & test_set)}")


def build_fold_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    heldout_id: str,
    label_map: Dict[int, int],
    cfg: Config,
) -> FoldData:
    train_ids = [sid for sid in subject_arrays.keys() if sid != heldout_id]

    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    heldout_subject: Dict[str, SubjectPartition] = {}

    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        if sid == heldout_id:
            packaged = build_subject_partitions_heldout(X, y, cfg)
            heldout_subject = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=True)
        else:
            packaged = build_subject_partitions_train(X, y, cfg)
            train_subjects[sid] = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in train_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across non-heldout subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in train_subjects.keys():
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for split_name in heldout_subject.keys():
        heldout_subject[split_name] = transform_partition(heldout_subject[split_name])

    for sid in train_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    for split_name in ("support", "adapt_val", "test"):
        if len(heldout_subject[split_name].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(
                f"Held-out {heldout_id} split {split_name} has too few windows "
                f"({len(heldout_subject[split_name].windows)})."
            )

    feat_dim = next(iter(subject_arrays.values()))[0].shape[1]
    return FoldData(
        heldout_id=heldout_id,
        train_ids=train_ids,
        train_subjects=train_subjects,
        heldout_subject=heldout_subject,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
        scaler=scaler,
    )


def build_fold_summary(fold: FoldData) -> Dict[str, Dict[str, int]]:
    out: Dict[str, Dict[str, int]] = {}
    for sid in fold.train_ids:
        out[sid] = {
            "train": int(len(fold.train_subjects[sid]["train"].labels)),
            "val": int(len(fold.train_subjects[sid]["val"].labels)),
        }
    out[fold.heldout_id] = {
        "support": int(len(fold.heldout_subject["support"].labels)),
        "adapt_val": int(len(fold.heldout_subject["adapt_val"].labels)),
        "test": int(len(fold.heldout_subject["test"].labels)),
    }
    return out


def inverse_label_map(label_map: Dict[int, int]) -> Dict[int, int]:
    return {int(v): int(k) for k, v in label_map.items()}


def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    return obj


def save_json(path: Union[str, Path], payload: Dict[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(payload), f, indent=2)


def scaler_to_state(scaler: StandardScaler) -> Dict[str, np.ndarray]:
    return {
        "mean": np.asarray(scaler.mean_, dtype=np.float32),
        "scale": np.asarray(scaler.scale_, dtype=np.float32),
        "var": np.asarray(scaler.var_, dtype=np.float32),
        "n_features_in": int(scaler.n_features_in_),
    }


def apply_saved_scaler(windows: np.ndarray, scaler_state: Dict[str, Any]) -> np.ndarray:
    mean = np.asarray(scaler_state["mean"], dtype=np.float32)
    scale = np.asarray(scaler_state["scale"], dtype=np.float32)
    scale = np.where(scale == 0, 1.0, scale).astype(np.float32)

    shape = windows.shape
    flat = windows.reshape(-1, shape[-1]).astype(np.float32)
    flat = (flat - mean) / scale
    return flat.reshape(shape).astype(np.float32)


def prepare_sensor_dataframe_for_inference(sensor_df: pd.DataFrame) -> pd.DataFrame:
    req_cols = feature_columns()
    missing = [c for c in req_cols if c not in sensor_df.columns]
    if missing:
        raise ValueError(
            "Input sensor dataframe is missing required columns: " + ", ".join(missing)
        )

    df = sensor_df.copy()
    df = df[req_cols]
    df = df.replace(-1.0, np.nan)
    df = df.interpolate(method="linear", limit_direction="both", axis=0)
    df = df.ffill().bfill()
    df = df.fillna(df.median())
    return df


def sensor_df_to_windows(sensor_df: pd.DataFrame, cfg: Config) -> Tuple[np.ndarray, np.ndarray]:
    df = prepare_sensor_dataframe_for_inference(sensor_df)
    X = df.to_numpy(dtype=np.float32)

    windows: List[np.ndarray] = []
    starts: List[int] = []
    for start in range(0, max(0, len(X) - cfg.seq_len + 1), cfg.stride):
        end = start + cfg.seq_len
        if end <= len(X):
            windows.append(X[start:end])
            starts.append(start)

    if not windows:
        return np.empty((0, cfg.seq_len, X.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.stack(windows).astype(np.float32), np.asarray(starts, dtype=np.int64)


def save_fold_artifacts(result: Dict[str, Any], cfg: Config) -> str:
    fold_dir = Path(cfg.save_dir) / f"fold_{result['heldout_id']}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    bundle = result["artifact_bundle"]
    torch.save(bundle, fold_dir / "inference_bundle.pt")

    save_json(fold_dir / "metrics.json", {
        "heldout_id": result["heldout_id"],
        "global_test": result["global_test"],
        "personalized_test": result["personalized_test"],
        "adapt_val": result["adapt_val"],
        "adapt_gate": result["adapt_gate"],
    })
    save_json(fold_dir / "history.json", {"history": result["history"]})
    save_json(fold_dir / "fold_summary.json", result["summary"])

    pd.DataFrame(result["history"]).to_csv(fold_dir / "training_history.csv", index=False)
    return str(fold_dir)


def save_experiment_summary(
    cfg: Config,
    all_results: List[Dict[str, Any]],
    global_mean: Dict[str, float],
    global_std: Dict[str, float],
    pers_mean: Dict[str, float],
    pers_std: Dict[str, float],
    label_map: Dict[int, int],
) -> None:
    payload = {
        "config": asdict(cfg),
        "label_map": {int(k): int(v) for k, v in label_map.items()},
        "idx_to_activity": inverse_label_map(label_map),
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "folds": [
            {
                "heldout_id": res["heldout_id"],
                "global_test": res["global_test"],
                "personalized_test": res["personalized_test"],
                "adapt_val": res["adapt_val"],
                "adapt_gate": res["adapt_gate"],
            }
            for res in all_results
        ],
    }
    save_json(Path(cfg.save_dir) / "loso_summary.json", payload)


def load_inference_bundle(bundle_path: Union[str, Path], map_location: str = "cpu") -> Dict[str, Any]:
    bundle = torch.load(bundle_path, map_location=map_location)
    return bundle


def build_model_from_bundle(bundle: Dict[str, Any], device: Union[str, torch.device] = "cpu") -> Tuple[nn.Module, Config, torch.device]:
    cfg = Config(**bundle["config"])
    device = torch.device(device)
    model = TemporalEncoder(bundle["feature_dim"], cfg).to(device)
    model.load_state_dict(bundle["model_state"])
    model.eval()
    return model, cfg, device


@torch.inference_mode()
def predict_windows_with_bundle(
    windows: np.ndarray,
    bundle_path: Union[str, Path],
    device: str = "cpu",
    use_personalized_head: bool = False,
    batch_size: int = 256,
) -> pd.DataFrame:
    bundle = load_inference_bundle(bundle_path, map_location=device)
    model, cfg, device = build_model_from_bundle(bundle, device=device)

    if windows.ndim != 3:
        raise ValueError(f"Expected windows shape [N, T, F], got {windows.shape}")

    windows_scaled = apply_saved_scaler(windows, bundle["scaler_state"])
    dummy_labels = np.zeros((len(windows_scaled),), dtype=np.int64)
    loader = DataLoader(
        WindowDataset(windows_scaled, dummy_labels),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

    num_classes = int(bundle["num_classes"])
    client_A = bundle["selected_A"] if use_personalized_head else torch.zeros(num_classes, cfg.proto_rank)
    log_tau = bundle["selected_log_tau"] if use_personalized_head else bundle["mean_log_tau"]

    probs, _ = predict_loader(
        model=model,
        loader=loader,
        global_proto=bundle["global_proto"].to(device),
        proto_basis=bundle["proto_basis"].to(device),
        client_A=client_A.to(device),
        log_tau=log_tau.to(device),
        device=device,
        cfg=cfg,
        use_amp=False,
    )

    pred_idx = probs.argmax(axis=1)
    pred_conf = probs.max(axis=1)
    idx_to_activity = {int(k): int(v) for k, v in bundle["idx_to_activity"].items()}
    pred_activity = [idx_to_activity[int(i)] for i in pred_idx]

    out = pd.DataFrame({
        "window_index": np.arange(len(pred_idx)),
        "pred_class_idx": pred_idx.astype(int),
        "pred_activity_id": pred_activity,
        "confidence": pred_conf.astype(float),
    })
    return out


def predict_sensor_dataframe(
    sensor_df: pd.DataFrame,
    bundle_path: Union[str, Path],
    device: str = "cpu",
    use_personalized_head: bool = False,
) -> pd.DataFrame:
    bundle = load_inference_bundle(bundle_path, map_location=device)
    cfg = Config(**bundle["config"])
    windows, starts = sensor_df_to_windows(sensor_df, cfg)
    if len(windows) == 0:
        raise ValueError(
            f"Not enough rows to create even one window. Need at least seq_len={cfg.seq_len} rows."
        )

    preds = predict_windows_with_bundle(
        windows=windows,
        bundle_path=bundle_path,
        device=device,
        use_personalized_head=use_personalized_head,
        batch_size=cfg.batch_size,
    )
    preds.insert(1, "start_row", starts.astype(int))
    preds.insert(2, "end_row", (starts + cfg.seq_len).astype(int))
    return preds


def make_loader(
    part: SubjectPartition,
    batch_size: int,
    shuffle: bool,
    cfg: Config,
) -> DataLoader:
    ds = WindowDataset(part.windows, part.labels)
    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=(cfg.pin_memory and str(cfg.device).startswith("cuda")),
        persistent_workers=(cfg.num_workers > 0),
        drop_last=False,
    )
    return DataLoader(ds, **loader_kwargs)


def fold_to_loaders(
    fold: FoldData,
    cfg: Config,
) -> Dict[str, object]:
    loaders: Dict[str, object] = {
        "train_clients": {},
        "heldout": {},
        "num_classes": fold.num_classes,
        "feature_dim": fold.feature_dim,
        "heldout_id": fold.heldout_id,
        "train_ids": fold.train_ids,
        "label_map": fold.label_map,
        "scaler": fold.scaler,
    }

    for sid in fold.train_ids:
        loaders["train_clients"][sid] = {
            "train": make_loader(fold.train_subjects[sid]["train"], cfg.batch_size, True, cfg),
            "val": make_loader(fold.train_subjects[sid]["val"], cfg.batch_size, False, cfg),
            "n_train": len(fold.train_subjects[sid]["train"].labels),
        }

    for split_name in ("support", "adapt_val", "test"):
        bs = cfg.personal_batch_size if split_name != "test" else cfg.batch_size
        loaders["heldout"][split_name] = make_loader(
            fold.heldout_subject[split_name], bs, split_name == "support", cfg
        )

    loaders["summary"] = build_fold_summary(fold)
    return loaders

## Model

Temporal encoder architecture: convolutional temporal stem, sinusoidal positional encoding, Transformer encoder, attentive pooling, and L2-normalized embeddings.

In [ ]:
# ============================================================
# Model
# ============================================================

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TemporalStem(nn.Module):
    def __init__(self, in_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, d_model, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=1, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.net(x)
        return x.transpose(1, 2)


class AttentivePool(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn = torch.softmax(self.score(x).squeeze(-1), dim=1)
        pooled = torch.sum(attn.unsqueeze(-1) * x, dim=1)
        return pooled


class TemporalEncoder(nn.Module):
    def __init__(self, in_dim: int, cfg: Config):
        super().__init__()
        self.stem = TemporalStem(in_dim, cfg.d_model, cfg.dropout)
        self.pos = SinusoidalPositionalEncoding(cfg.d_model, max_len=max(2048, cfg.seq_len + 8))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.ff_dim,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=False,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.pool = AttentivePool(cfg.d_model)
        self.out_norm = nn.LayerNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        h = self.pos(h)
        h = self.encoder(h)
        h = self.pool(h)
        h = self.out_norm(h)
        z = self.proj(h)
        z = F.normalize(z, dim=-1)
        return z

## Prototype utilities

Prototype deformation, cosine-style logits, client loss, class weighting, and client-specific temperature utilities.

In [ ]:
# ============================================================
# Prototype utilities
# ============================================================

def orthonormal_random(d: int, r: int, device: torch.device) -> torch.Tensor:
    q, _ = torch.linalg.qr(torch.randn(d, r, device=device))
    return q[:, :r].contiguous()


def personalized_prototypes_ccd(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
) -> torch.Tensor:
    proto = global_proto + client_A @ proto_basis.T
    proto = F.normalize(proto, dim=-1)
    return proto


def proto_logits(
    z: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
) -> torch.Tensor:
    P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
    tau = torch.exp(log_tau).clamp(cfg.tau_min, cfg.tau_max)
    return tau * (z @ P_i.T)


def local_loss(
    z: torch.Tensor,
    y: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    class_weight: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
    logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)

    loss_proto = F.cross_entropy(
        logits,
        y,
        weight=class_weight,
        label_smoothing=0.05,
    )
    align = 1.0 - torch.sum(z * P_i[y], dim=1).mean()
    reg_A = client_A.pow(2).mean()
    reg_tau = (log_tau - math.log(cfg.tau_init)) ** 2

    loss = (
        loss_proto
        + cfg.lambda_align * align
        + cfg.lambda_A * reg_A
        + cfg.lambda_tau * reg_tau
    )

    stats = {
        "loss_proto": float(loss_proto.detach().item()),
        "loss_align": float(align.detach().item()),
        "loss_regA": float(reg_A.detach().item()),
        "loss_regTau": float(reg_tau.detach().item()),
        "tau": float(torch.exp(log_tau.detach()).clamp(cfg.tau_min, cfg.tau_max).item()),
    }
    return loss, stats


def make_class_weight_from_loader(
    train_loader: DataLoader,
    num_classes: int,
    device: torch.device,
    cfg: Config,
) -> torch.Tensor:
    labels = train_loader.dataset.y.detach().cpu().numpy()
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)

    present = counts > 0
    weights = np.ones(num_classes, dtype=np.float32)

    if present.any():
        ref = float(np.median(counts[present]))
        weights[present] = np.power(ref / np.clip(counts[present], 1.0, None), cfg.class_weight_power)
        weights[present] = np.clip(weights[present], 1.0 / cfg.class_weight_max, cfg.class_weight_max)
        weights[present] /= max(weights[present].mean(), 1e-8)

    return torch.tensor(weights, dtype=torch.float32, device=device)

## Evaluation

Prediction and evaluation functions for computing metrics from dataloaders.

In [ ]:
# ============================================================
# Evaluation
# ============================================================

@torch.inference_mode()
def predict_loader(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    device: torch.device,
    cfg: Config,
    use_amp: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = use_amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = model(xb)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())

    if not probs_all:
        num_classes = global_proto.shape[0]
        return np.empty((0, num_classes), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    device: torch.device,
) -> Dict[str, float]:
    probs, labels = predict_loader(
        model, loader, global_proto, proto_basis, client_A, log_tau, device, cfg, cfg.amp
    )
    return summarize_probs(probs, labels, num_classes=global_proto.shape[0], ece_bins=cfg.ece_bins)

## Client training and prototype extraction

Local client training, mixed precision training, confidence-weighted empirical prototype extraction, and client parameter updates.

In [ ]:
# ============================================================
# Client training and prototype extraction
# ============================================================

def clone_model(model: nn.Module) -> nn.Module:
    return copy.deepcopy(model)


def train_one_client(
    global_model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    A_init: torch.Tensor,
    log_tau_init: torch.Tensor,
    cfg: Config,
    device: torch.device,
    warmup_extract: bool = False,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    del val_loader  # local validation was unused by the outer loop and only added runtime

    local_model = clone_model(global_model).to(device)

    client_A = nn.Parameter(A_init.clone().to(device))
    log_tau = nn.Parameter(log_tau_init.clone().to(device))
    class_weight = make_class_weight_from_loader(train_loader, global_proto.shape[0], device, cfg)

    optimizer = torch.optim.AdamW(
        list(local_model.parameters()) + [client_A, log_tau],
        lr=cfg.lr_encoder,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.local_epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    optimizer_steps = 0
    scheduler_steps = 0

    for _epoch in range(cfg.local_epochs):
        local_model.train()

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = local_model(xb)
                loss, _ = local_loss(
                    z, yb, global_proto, proto_basis, client_A, log_tau, cfg, class_weight=class_weight
                )

            if amp_enabled:
                scaler.scale(loss).backward()

                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        list(local_model.parameters()) + [client_A, log_tau],
                        cfg.grad_clip,
                    )

                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                new_scale = scaler.get_scale()

                if new_scale >= old_scale:
                    optimizer_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(
                        list(local_model.parameters()) + [client_A, log_tau],
                        cfg.grad_clip,
                    )
                optimizer.step()
                optimizer_steps += 1

        if optimizer_steps > scheduler_steps:
            scheduler.step()
            scheduler_steps += 1

    proto_summary = {}
    emp, present, counts = extract_confidence_weighted_prototypes(
        model=local_model,
        loader=train_loader,
        global_proto=global_proto,
        proto_basis=proto_basis,
        client_A=client_A.detach(),
        log_tau=log_tau.detach(),
        num_classes=global_proto.shape[0],
        emb_dim=cfg.emb_dim,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=warmup_extract,
    )
    proto_summary["empirical"] = emp
    proto_summary["present"] = present
    proto_summary["counts"] = counts

    state_dict = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}

    del local_model

    return (
        state_dict,
        client_A.detach().cpu().clone(),
        log_tau.detach().cpu().clone(),
        proto_summary,
    )


@torch.inference_mode()
def extract_confidence_weighted_prototypes(
    model: nn.Module,
    loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    client_A: torch.Tensor,
    log_tau: torch.Tensor,
    num_classes: int,
    emb_dim: int,
    device: torch.device,
    cfg: Config,
    use_amp: bool = True,
    warmup: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    model.eval()

    proto_sum = torch.zeros(num_classes, emb_dim, device=device)
    weight_sum = torch.zeros(num_classes, device=device)
    counts = torch.zeros(num_classes, device=device)

    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = use_amp and device.type == "cuda"

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = model(xb)
            logits = proto_logits(z, global_proto, proto_basis, client_A, log_tau, cfg)
            probs = torch.softmax(logits, dim=1)

        if warmup:
            weights = torch.ones_like(yb, dtype=z.dtype, device=device)
        else:
            true_conf = probs.gather(1, yb.unsqueeze(1)).squeeze(1)
            max_conf = probs.max(dim=1).values
            weights = 0.7 * true_conf + 0.3 * max_conf

        proto_sum.index_add_(0, yb, z * weights.unsqueeze(1))
        weight_sum.index_add_(0, yb, weights)
        counts.index_add_(0, yb, torch.ones_like(weights))

    empirical = torch.zeros_like(proto_sum)
    present = (weight_sum > 0) & (counts >= cfg.min_proto_samples_per_class)

    empirical[present] = proto_sum[present] / weight_sum[present].unsqueeze(1)
    empirical[present] = F.normalize(empirical[present], dim=1)

    return empirical.detach().cpu(), present.detach().cpu(), counts.detach().cpu()

## Server aggregation and geometry update

Federated averaging plus shared prototype, deformation basis, and closed-form client coefficient updates.

In [ ]:
# ============================================================
# Server aggregation and geometry update
# ============================================================

def average_state_dicts(
    state_dicts: List[Dict[str, torch.Tensor]],
    weights: List[float],
) -> Dict[str, torch.Tensor]:
    total = float(sum(weights))
    weights = [w / total for w in weights]
    out: Dict[str, torch.Tensor] = {}
    keys = state_dicts[0].keys()
    for k in keys:
        ref = state_dicts[0][k]
        if torch.is_floating_point(ref):
            acc = None
            for sd, w in zip(state_dicts, weights):
                tensor = sd[k].float()
                acc = tensor * w if acc is None else acc + tensor * w
            out[k] = acc.to(dtype=ref.dtype)
        else:
            out[k] = ref.clone()
    return out


def solve_A_closed_form(
    empirical_proto: torch.Tensor,
    present: torch.Tensor,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    lambda_A: float,
) -> torch.Tensor:
    BtB = proto_basis.T @ proto_basis
    r = BtB.shape[0]
    inv = torch.linalg.inv(BtB + lambda_A * torch.eye(r, device=proto_basis.device, dtype=proto_basis.dtype))
    A = (empirical_proto - global_proto) @ proto_basis @ inv
    A = A * present.float().unsqueeze(1)
    return A


def server_geometry_update(
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    proto_summaries: Dict[str, Dict[str, torch.Tensor]],
    cfg: Config,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, torch.Tensor]]:
    """
    Minimizes approximately:
        sum_{i,c} w_{i,c} || H_ic - p_c - B a_ic ||^2
        + lambda_A sum_i ||A_i||^2 + lambda_B ||B^T B - I||^2
    """
    old_P = global_proto.clone().to(device)
    old_B = proto_basis.clone().to(device)

    P = old_P.clone()
    B = old_B.clone()

    client_ids = list(proto_summaries.keys())
    H = torch.stack([proto_summaries[sid]["empirical"].to(device) for sid in client_ids], dim=0)
    M = torch.stack([proto_summaries[sid]["present"].to(device) for sid in client_ids], dim=0).bool()
    W = torch.stack([proto_summaries[sid]["counts"].to(device) for sid in client_ids], dim=0).float()
    W = torch.where(M, torch.sqrt(torch.clamp(W, min=0.0)), torch.zeros_like(W))

    K, C, d = H.shape
    r = B.shape[1]

    A_list = []
    for i in range(K):
        A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
        A_list.append(A_i)
    A = torch.stack(A_list, dim=0)

    for _ in range(cfg.server_alt_iters):
        numer = torch.zeros_like(P)
        denom = torch.zeros(C, device=device)
        for i in range(K):
            recon_free = H[i] - A[i] @ B.T
            numer += W[i].unsqueeze(1) * recon_free
            denom += W[i]

        keep = denom > 0
        P[keep] = numer[keep] / denom[keep].unsqueeze(1)
        P = F.normalize(P, dim=1)

        B_param = nn.Parameter(B.clone())
        opt = torch.optim.Adam([B_param], lr=cfg.server_geom_lr)

        for _step in range(cfg.server_geom_steps):
            opt.zero_grad(set_to_none=True)
            recon = P.unsqueeze(0) + torch.matmul(A, B_param.T)
            sq = ((H - recon) ** 2).sum(dim=2)
            data_term = (W * sq).sum() / (W.sum() + 1e-8)
            orth = ((B_param.T @ B_param) - torch.eye(r, device=device)).pow(2).mean()
            loss = data_term + cfg.lambda_B * orth
            loss.backward()
            opt.step()

        with torch.no_grad():
            q, _ = torch.linalg.qr(B_param.data)
            B = q[:, :r].contiguous()

        A_list = []
        for i in range(K):
            A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
            A_list.append(A_i)
        A = torch.stack(A_list, dim=0)

    # momentum smoothing of shared geometry
    P = F.normalize(cfg.server_proto_momentum * old_P + (1.0 - cfg.server_proto_momentum) * P, dim=1)
    B_blend = cfg.server_basis_momentum * old_B + (1.0 - cfg.server_basis_momentum) * B
    q, _ = torch.linalg.qr(B_blend)
    B = q[:, :r].contiguous()

    A_list = []
    for i in range(K):
        A_i = solve_A_closed_form(H[i], M[i], P, B, cfg.lambda_A)
        A_list.append(A_i)
    A = torch.stack(A_list, dim=0)

    client_A = {sid: A[idx].detach().cpu().clone() for idx, sid in enumerate(client_ids)}
    return P.detach(), B.detach(), client_A

## Held-out personalization

Support-set adaptation for an unseen held-out subject with validation gating to avoid harmful personalization.

In [ ]:
# ============================================================
# Held-out personalization
# ============================================================

def adapt_heldout_client(
    model: nn.Module,
    support_loader: DataLoader,
    adapt_val_loader: DataLoader,
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    cfg: Config,
    device: torch.device,
    init_log_tau: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float], Dict[str, object]]:
    frozen_model = clone_model(model).to(device)
    frozen_model.eval()
    for p in frozen_model.parameters():
        p.requires_grad = False

    C, d = global_proto.shape
    r = proto_basis.shape[1]
    global_proto = global_proto.to(device)
    proto_basis = proto_basis.to(device)

    if init_log_tau is None:
        init_log_tau_device = torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32, device=device))
    else:
        init_log_tau_device = init_log_tau.detach().to(device).float()

    baseline_A = torch.zeros(C, r, dtype=torch.float32, device=device)

    # Global/no-personalization baseline on adapt_val
    baseline_metrics = evaluate_loader(
        frozen_model,
        adapt_val_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        cfg,
        device,
    )
    baseline_score = personalization_selection_score(baseline_metrics)

    # Closed-form init from support set
    init_emp, present, _ = extract_confidence_weighted_prototypes(
        frozen_model,
        support_loader,
        global_proto,
        proto_basis,
        baseline_A,
        init_log_tau_device,
        num_classes=C,
        emb_dim=d,
        device=device,
        cfg=cfg,
        use_amp=cfg.amp,
        warmup=True,
    )
    init_A = solve_A_closed_form(
        init_emp.to(device),
        present.to(device),
        global_proto,
        proto_basis,
        cfg.lambda_A,
    ).detach()

    client_A = nn.Parameter(init_A.clone())
    log_tau = nn.Parameter(init_log_tau_device.clone())

    init_metrics = evaluate_loader(
        frozen_model,
        adapt_val_loader,
        global_proto,
        proto_basis,
        client_A.detach(),
        log_tau.detach(),
        cfg,
        device,
    )

    hps = choose_personalization_hparams(init_metrics["f1"], cfg)
    adapt_epochs = int(hps["epochs"])
    warm_epochs = int(hps["warm_epochs"])
    anchor_w = float(hps["anchor_w"])
    lr_A = float(hps["lr_A"])
    lr_tau = float(hps["lr_tau"])

    optimizer_A = torch.optim.AdamW(
        [{"params": [client_A], "lr": lr_A, "weight_decay": 1e-4}]
    )
    optimizer_joint = torch.optim.AdamW(
        [
            {"params": [client_A], "lr": lr_A, "weight_decay": 1e-4},
            {"params": [log_tau], "lr": lr_tau, "weight_decay": 0.0},
        ]
    )

    best = {
        "A": client_A.detach().cpu().clone(),
        "log_tau": log_tau.detach().cpu().clone(),
        "score": personalization_selection_score(init_metrics),
        "metrics": init_metrics,
    }

    no_improve = 0
    init_A_device = init_A.detach()

    for epoch in range(adapt_epochs):
        frozen_model.eval()
        optimizer = optimizer_A if epoch < warm_epochs else optimizer_joint

        for xb, yb in support_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            z = frozen_model(xb)
            P_i = personalized_prototypes_ccd(global_proto, proto_basis, client_A)
            logits = proto_logits(
                z,
                global_proto,
                proto_basis,
                client_A,
                log_tau,
                cfg,
            )

            loss_proto = F.cross_entropy(logits, yb, label_smoothing=0.02)
            align = 1.0 - torch.sum(z * P_i[yb], dim=1).mean()
            reg_A = client_A.pow(2).mean()
            anchor_A = (client_A - init_A_device).pow(2).mean()
            anchor_tau = (log_tau - init_log_tau_device).pow(2)

            loss = (
                loss_proto
                + cfg.lambda_align * align
                + cfg.lambda_A * reg_A
                + anchor_w * anchor_A
                + 0.5 * cfg.lambda_tau * anchor_tau
            )

            loss.backward()

            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                if epoch < warm_epochs:
                    torch.nn.utils.clip_grad_norm_([client_A], cfg.grad_clip)
                else:
                    torch.nn.utils.clip_grad_norm_([client_A, log_tau], cfg.grad_clip)

            optimizer.step()

        val_metrics = evaluate_loader(
            frozen_model,
            adapt_val_loader,
            global_proto,
            proto_basis,
            client_A.detach(),
            log_tau.detach(),
            cfg,
            device,
        )

        score = personalization_selection_score(val_metrics)

        if score > best["score"]:
            best["score"] = score
            best["A"] = client_A.detach().cpu().clone()
            best["log_tau"] = log_tau.detach().cpu().clone()
            best["metrics"] = val_metrics
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg.personalization_patience:
            break

    required_margin = cfg.personalization_gate_score_margin
    if baseline_metrics["f1"] >= cfg.easy_subject_f1_threshold:
        required_margin += cfg.personalization_gate_score_margin_easy

    use_personalization = (
        best["score"] > baseline_score + required_margin
        and best["metrics"]["f1"] >= baseline_metrics["f1"] + cfg.personalization_gate_f1_margin
    )

    if use_personalization:
        selected_A = best["A"]
        selected_log_tau = best["log_tau"]
        selected_metrics = best["metrics"]
    else:
        selected_A = baseline_A.detach().cpu().clone()
        selected_log_tau = init_log_tau_device.detach().cpu().clone()
        selected_metrics = baseline_metrics

    gate_info = {
        "used_personalization": bool(use_personalization),
        "baseline_metrics": baseline_metrics,
        "baseline_score": float(baseline_score),
        "personalized_metrics": best["metrics"],
        "personalized_score": float(best["score"]),
        "required_margin": float(required_margin),
    }

    return selected_A, selected_log_tau, selected_metrics, gate_info

## Fold training

One-fold LOSO federated training loop, round tracking, best-snapshot selection, and fold-level artifacts.

In [ ]:
# ============================================================
# Fold training
# ============================================================

def train_one_fold(
    loaders: Dict[str, object],
    cfg: Config,
    device: torch.device,
) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    global_model = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    proto_basis = orthonormal_random(cfg.emb_dim, cfg.proto_rank, device)

    client_A = {sid: torch.zeros(num_classes, cfg.proto_rank, dtype=torch.float32) for sid in train_ids}
    client_log_tau = {
        sid: torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32))
        for sid in train_ids
    }

    best_snapshot = None
    best_score = -1.0
    history: List[Dict[str, float]] = []

    warmup_rounds = min(3, max(1, cfg.rounds // 4))
    round_bar = tqdm(range(1, cfg.rounds + 1), desc=f"Fold {heldout_id}", leave=True)

    for rnd in round_bar:
        local_state_dicts = []
        local_weights = []
        proto_summaries: Dict[str, Dict[str, torch.Tensor]] = {}

        for sid in train_ids:
            state_dict, A_local, log_tau_local, proto_summary = train_one_client(
                global_model=global_model,
                train_loader=train_clients[sid]["train"],
                val_loader=train_clients[sid]["val"],
                global_proto=global_proto,
                proto_basis=proto_basis,
                A_init=client_A[sid],
                log_tau_init=client_log_tau[sid],
                cfg=cfg,
                device=device,
                warmup_extract=(rnd <= warmup_rounds),
            )

            local_state_dicts.append(state_dict)
            local_weights.append(train_clients[sid]["n_train"])
            proto_summaries[sid] = proto_summary
            client_log_tau[sid] = log_tau_local.clone()

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        global_proto, proto_basis, new_client_A = server_geometry_update(
            global_proto=global_proto,
            proto_basis=proto_basis,
            proto_summaries=proto_summaries,
            cfg=cfg,
            device=device,
        )
        client_A = new_client_A

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        per_client_val_sizes: Dict[str, int] = {}

        with torch.no_grad():
            for sid in train_ids:
                metrics = evaluate_loader(
                    global_model,
                    train_clients[sid]["val"],
                    global_proto,
                    proto_basis,
                    client_A[sid].to(device),
                    client_log_tau[sid].to(device),
                    cfg,
                    device,
                )
                per_client_metrics[sid] = metrics
                per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(
            per_client_metrics, per_client_val_sizes, cfg
        )

        round_bar.set_postfix({
            "val_f1": f"{mean_f1:.4f}",
            "val_acc": f"{mean_acc:.4f}",
            "val_ece": f"{mean_ece:.4f}",
        })

        history.append({
            "round": rnd,
            "mean_val_f1": mean_f1,
            "mean_val_acc": mean_acc,
            "mean_val_ece": mean_ece,
        })

        if score > best_score:
            best_score = score
            best_snapshot = {
                "model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()},
                "global_proto": global_proto.detach().cpu().clone(),
                "proto_basis": proto_basis.detach().cpu().clone(),
                "client_A": {sid: a.detach().cpu().clone() for sid, a in client_A.items()},
                "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in client_log_tau.items()},
                "history": copy.deepcopy(history),
            }

    assert best_snapshot is not None

    global_model.load_state_dict(best_snapshot["model_state"])
    global_proto = best_snapshot["global_proto"].to(device)
    proto_basis = best_snapshot["proto_basis"].to(device)
    saved_log_tau = best_snapshot["client_log_tau"]

    train_size_weights = torch.tensor(
        [train_clients[sid]["n_train"] for sid in train_ids],
        dtype=torch.float32,
        device=device,
    )
    tau_stack = torch.stack([saved_log_tau[sid].float().to(device) for sid in train_ids], dim=0)
    mean_log_tau = (train_size_weights * tau_stack).sum() / train_size_weights.sum()

    global_test = evaluate_loader(
        global_model,
        heldout["test"],
        global_proto,
        proto_basis,
        torch.zeros(num_classes, cfg.proto_rank, device=device),
        mean_log_tau,
        cfg,
        device,
    )

    best_A, best_log_tau, adapt_val_metrics, adapt_gate = adapt_heldout_client(
        model=global_model,
        support_loader=heldout["support"],
        adapt_val_loader=heldout["adapt_val"],
        global_proto=global_proto,
        proto_basis=proto_basis,
        cfg=cfg,
        device=device,
        init_log_tau=mean_log_tau.detach(),
    )

    personalized_test = evaluate_loader(
        global_model,
        heldout["test"],
        global_proto,
        proto_basis,
        best_A.to(device),
        best_log_tau.to(device),
        cfg,
        device,
    )

    return {
        "heldout_id": heldout_id,
        "summary": loaders["summary"],
        "history": best_snapshot["history"],
        "global_test": global_test,
        "personalized_test": personalized_test,
        "adapt_val": adapt_val_metrics,
        "adapt_gate": adapt_gate,
        "artifact_bundle": {
            "heldout_id": heldout_id,
            "feature_dim": feature_dim,
            "num_classes": num_classes,
            "config": asdict(cfg),
            "model_state": {k: v.detach().cpu().clone() for k, v in best_snapshot["model_state"].items()},
            "global_proto": best_snapshot["global_proto"].detach().cpu().clone(),
            "proto_basis": best_snapshot["proto_basis"].detach().cpu().clone(),
            "mean_log_tau": mean_log_tau.detach().cpu().clone(),
            "selected_A": best_A.detach().cpu().clone(),
            "selected_log_tau": best_log_tau.detach().cpu().clone(),
            "label_map": {int(k): int(v) for k, v in loaders["label_map"].items()},
            "idx_to_activity": inverse_label_map(loaders["label_map"]),
            "feature_columns": dataset_feature_columns(cfg),
            "scaler_state": scaler_to_state(loaders["scaler"]),
        },
    }

## Dataset adapter: PAMAP2 or UCI HAR

Dataset loader dispatch and utilities that support both PAMAP2 and UCI HAR-style data.

In [ ]:
# ============================================================
# Dataset adapter: PAMAP2 or UCI HAR
# ============================================================

UCI_HAR_FEATURE_NAMES: Tuple[str, ...] = (
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
)

UCI_HAR_ACTIVITY_NAMES: Dict[int, str] = {
    1: "WALKING",
    2: "WALKING_UPSTAIRS",
    3: "WALKING_DOWNSTAIRS",
    4: "SITTING",
    5: "STANDING",
    6: "LAYING",
}


def dataset_feature_columns(cfg: Config) -> List[str]:
    if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har":
        return list(UCI_HAR_FEATURE_NAMES)
    return feature_columns()


def find_uci_har_root(data_dir: Union[str, Path]) -> Path:
    """Return the folder containing the official UCI HAR train/ and test/ subfolders."""
    root = Path(data_dir)

    def is_uci_root(p: Path) -> bool:
        return (
            (p / "train" / "Inertial Signals").exists()
            and (p / "test" / "Inertial Signals").exists()
            and (p / "train" / "y_train.txt").exists()
            and (p / "test" / "y_test.txt").exists()
        )

    if is_uci_root(root):
        return root

    # Common case after unzipping: /.../UCI HAR Dataset/UCI HAR Dataset
    nested = root / "UCI HAR Dataset"
    if is_uci_root(nested):
        return nested

    # Fallback: search under the supplied directory first, then MyDrive.
    search_roots = [root, Path("/content/drive/MyDrive")]
    seen = set()
    for base in search_roots:
        if not base.exists() or str(base) in seen:
            continue
        seen.add(str(base))
        for p in base.rglob("Inertial Signals"):
            candidate = p.parent.parent
            if is_uci_root(candidate):
                return candidate

    raise FileNotFoundError(
        "Could not find the UCI HAR Dataset folder. Set cfg.data_dir to the folder containing "
        "train/Inertial Signals and test/Inertial Signals. Current data_dir=" + repr(str(data_dir))
    )


def load_uci_split(root: Path, split: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Load one official UCI HAR split as X=[N,128,9], y=[N], subject=[N]."""
    split_dir = root / split
    inertial_dir = split_dir / "Inertial Signals"
    suffix = "train" if split == "train" else "test"

    signal_files = [
        f"body_acc_x_{suffix}.txt",
        f"body_acc_y_{suffix}.txt",
        f"body_acc_z_{suffix}.txt",
        f"body_gyro_x_{suffix}.txt",
        f"body_gyro_y_{suffix}.txt",
        f"body_gyro_z_{suffix}.txt",
        f"total_acc_x_{suffix}.txt",
        f"total_acc_y_{suffix}.txt",
        f"total_acc_z_{suffix}.txt",
    ]

    signals = []
    for fname in signal_files:
        fp = inertial_dir / fname
        if not fp.exists():
            raise FileNotFoundError(f"Missing UCI HAR signal file: {fp}")
        arr = np.loadtxt(fp, dtype=np.float32)  # [N, 128]
        signals.append(arr)

    X = np.stack(signals, axis=-1).astype(np.float32)  # [N, 128, 9]
    y = np.loadtxt(split_dir / f"y_{suffix}.txt", dtype=np.int64).reshape(-1)
    subjects = np.loadtxt(split_dir / f"subject_{suffix}.txt", dtype=np.int64).reshape(-1)

    if X.shape[0] != len(y) or len(y) != len(subjects):
        raise RuntimeError(
            f"UCI HAR {split} length mismatch: X={X.shape[0]}, y={len(y)}, subjects={len(subjects)}"
        )
    return X, y, subjects


def load_uci_har_subject_arrays(cfg: Config) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray]], Dict[int, int]]:
    """
    Load UCI HAR raw inertial windows and group by subject.
    Returns subject_arrays[sid] = (windows [N,128,9], labels [N] zero-indexed).
    """
    root = find_uci_har_root(cfg.data_dir)
    print(f"[UCI HAR] Using dataset root: {root}")

    X_train, y_train, s_train = load_uci_split(root, "train")
    X_test, y_test, s_test = load_uci_split(root, "test")

    X_all = np.concatenate([X_train, X_test], axis=0).astype(np.float32)
    y_all_orig = np.concatenate([y_train, y_test], axis=0).astype(np.int64)
    s_all = np.concatenate([s_train, s_test], axis=0).astype(np.int64)

    label_map = {int(a): idx for idx, a in enumerate(sorted(np.unique(y_all_orig).tolist()))}
    y_all = np.asarray([label_map[int(a)] for a in y_all_orig], dtype=np.int64)

    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for sid_num in sorted(np.unique(s_all).tolist()):
        mask = s_all == sid_num
        sid = f"subject{int(sid_num):02d}"
        subject_arrays[sid] = (X_all[mask].astype(np.float32), y_all[mask].astype(np.int64))

    print(
        f"[UCI HAR] Loaded {len(subject_arrays)} subjects | "
        f"windows={len(y_all)} | shape={X_all.shape} | label_map={label_map}"
    )
    return subject_arrays, label_map


def load_dataset_subject_arrays(cfg: Config) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray]], Dict[int, int]]:
    dataset = getattr(cfg, "dataset_name", "pamap2").lower()
    if dataset == "uci_har":
        return load_uci_har_subject_arrays(cfg)
    if dataset == "pamap2":
        subject_files = discover_subject_files(cfg)
        subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
        label_map = build_global_label_map(subject_dfs)
        subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}
        return subject_arrays, label_map
    raise ValueError(f"Unknown dataset_name={cfg.dataset_name!r}. Use 'pamap2' or 'uci_har'.")


def split_counts_with_gap(total: int, ratios: List[float], min_counts: List[int], gap: int) -> Optional[List[Tuple[int, int]]]:
    """Split index positions [0,total) into k contiguous ranges with gaps between them."""
    k = len(ratios)
    usable = total - gap * (k - 1)
    if usable < sum(min_counts):
        return None

    lengths = allocate_lengths_with_minimum(usable, ratios, min_counts)
    if lengths is None:
        return None

    ranges: List[Tuple[int, int]] = []
    pos = 0
    for j, length in enumerate(lengths):
        ranges.append((pos, pos + length))
        pos += length
        if j < k - 1:
            pos += gap
    return ranges


def prewindowed_split_by_class(
    X: np.ndarray,
    y: np.ndarray,
    split_names: List[str],
    ratios: List[float],
    cfg: Config,
    train_mode: bool,
) -> Dict[str, SubjectPartition]:
    """
    Split already-windowed UCI HAR data by class, preserving local order.
    This keeps support/adapt/test and train/val class coverage whenever possible.
    """
    assert X.ndim == 3, f"Expected prewindowed X with shape [N,T,F], got {X.shape}"
    feat_dim = X.shape[-1]
    out: Dict[str, List[SubjectPartition]] = {name: [] for name in split_names}
    gap = int(getattr(cfg, "uci_purge_gap_windows", 0))

    classes = sorted(np.unique(y).tolist())
    for cls in classes:
        idx = np.where(y == cls)[0]
        n = len(idx)

        if train_mode:
            if n < 2:
                # Too few examples for validation; keep rare examples for local training.
                out["train"].append(SubjectPartition(X[idx], y[idx]))
                continue
            min_counts = [1, 1]
        else:
            if n < 3:
                # Too few examples for reliable adaptation; keep them for final testing only.
                out["test"].append(SubjectPartition(X[idx], y[idx]))
                continue
            min_counts = [1, 1, 1]

        ranges = split_counts_with_gap(n, ratios, min_counts=min_counts, gap=gap)
        if ranges is None:
            # Retry without a purge gap if the class is small.
            ranges = split_counts_with_gap(n, ratios, min_counts=min_counts, gap=0)
        if ranges is None:
            # Final conservative fallback.
            target = "train" if train_mode else "test"
            out[target].append(SubjectPartition(X[idx], y[idx]))
            continue

        for name, (a, b) in zip(split_names, ranges):
            chosen = idx[a:b]
            if len(chosen) > 0:
                out[name].append(SubjectPartition(X[chosen], y[chosen]))

    return {name: concat_partitions(parts, cfg.seq_len, feat_dim) for name, parts in out.items()}


def build_prewindowed_subject_partitions_train(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    return prewindowed_split_by_class(
        X=X,
        y=y,
        split_names=["train", "val"],
        ratios=[cfg.train_subject_train_ratio, cfg.train_subject_val_ratio],
        cfg=cfg,
        train_mode=True,
    )


def build_prewindowed_subject_partitions_heldout(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    return prewindowed_split_by_class(
        X=X,
        y=y,
        split_names=["support", "adapt_val", "test"],
        ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
        cfg=cfg,
        train_mode=False,
    )


# Override the original PAMAP2-only build_fold_data so it also accepts UCI HAR [N,T,F] windows.
def build_fold_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    heldout_id: str,
    label_map: Dict[int, int],
    cfg: Config,
) -> FoldData:
    train_ids = [sid for sid in subject_arrays.keys() if sid != heldout_id]

    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    heldout_subject: Dict[str, SubjectPartition] = {}

    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        if X.ndim == 3:
            packaged = (
                build_prewindowed_subject_partitions_heldout(X, y, cfg)
                if sid == heldout_id
                else build_prewindowed_subject_partitions_train(X, y, cfg)
            )
        else:
            packaged = (
                build_subject_partitions_heldout(X, y, cfg)
                if sid == heldout_id
                else build_subject_partitions_train(X, y, cfg)
            )

        if sid == heldout_id:
            heldout_subject = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=True)
        else:
            train_subjects[sid] = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in train_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across non-heldout subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in train_subjects.keys():
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for split_name in heldout_subject.keys():
        heldout_subject[split_name] = transform_partition(heldout_subject[split_name])

    for sid in train_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    for split_name in ("support", "adapt_val", "test"):
        if len(heldout_subject[split_name].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(
                f"Held-out {heldout_id} split {split_name} has too few windows "
                f"({len(heldout_subject[split_name].windows)})."
            )

    first_X = next(iter(subject_arrays.values()))[0]
    feat_dim = first_X.shape[-1] if first_X.ndim == 3 else first_X.shape[1]
    return FoldData(
        heldout_id=heldout_id,
        train_ids=train_ids,
        train_subjects=train_subjects,
        heldout_subject=heldout_subject,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
        scaler=scaler,
    )


def metric_mean_std_ci(values: List[float]) -> Dict[str, float]:
    vals = np.asarray(values, dtype=np.float64)
    n = len(vals)
    mean = float(np.mean(vals))
    std = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    if n > 1:
        try:
            from scipy import stats
            tcrit = float(stats.t.ppf(0.975, n - 1))
        except Exception:
            tcrit = 1.96
        half = float(tcrit * std / math.sqrt(n))
    else:
        half = 0.0
    return {
        "n": int(n),
        "mean": mean,
        "std": std,
        "ci95_low": mean - half,
        "ci95_high": mean + half,
    }


def save_loso_csv_tables(cfg: Config, all_results: List[Dict[str, Any]]) -> None:
    """Save per-subject results and paper-ready mean/std/95% CI tables."""
    out_dir = Path(cfg.save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    fold_rows: List[Dict[str, Any]] = []
    for res in all_results:
        row: Dict[str, Any] = {"heldout_id": res["heldout_id"]}
        for prefix, key in [("global", "global_test"), ("personalized", "personalized_test")]:
            for m, v in res[key].items():
                row[f"{prefix}_{m}"] = float(v)
        row["used_personalization"] = bool(res["adapt_gate"]["used_personalization"])
        row["baseline_score"] = float(res["adapt_gate"]["baseline_score"])
        row["personalized_score"] = float(res["adapt_gate"]["personalized_score"])
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(out_dir / f"{cfg.dataset_name}_cdpl_fold_results.csv", index=False)

    summary_rows: List[Dict[str, Any]] = []
    for variant, prefix in [("CDPL Global", "global"), ("CDPL Personalized", "personalized")]:
        for metric in ["acc", "f1", "ece", "brier"]:
            stats_dict = metric_mean_std_ci(fold_df[f"{prefix}_{metric}"].tolist())
            summary_rows.append({
                "dataset": cfg.dataset_name,
                "method": variant,
                "metric": metric,
                **stats_dict,
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / f"{cfg.dataset_name}_cdpl_summary_ci.csv", index=False)

    # Wide table for direct paper use.
    wide_rows: List[Dict[str, Any]] = []
    for variant, prefix in [("CDPL Global", "global"), ("CDPL Personalized", "personalized")]:
        row = {"dataset": cfg.dataset_name, "method": variant}
        for metric in ["acc", "f1", "ece", "brier"]:
            s = metric_mean_std_ci(fold_df[f"{prefix}_{metric}"].tolist())
            row[metric] = f"{s['mean']:.4f} ± {s['std']:.4f} [{s['ci95_low']:.4f}, {s['ci95_high']:.4f}]"
        wide_rows.append(row)
    pd.DataFrame(wide_rows).to_csv(out_dir / f"{cfg.dataset_name}_cdpl_paper_table.csv", index=False)


# Override the original run_loso_experiment to load either PAMAP2 or UCI HAR.
def run_loso_experiment(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_arrays, label_map = load_dataset_subject_arrays(cfg)

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS ================")
    print(subject_ids)
    print(f"Label map (activity_id -> class_idx): {label_map}")
    if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har":
        print(f"Activity names: {UCI_HAR_ACTIVITY_NAMES}")

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)

        print("Fold summary:")
        for sid, stats_dict in loaders["summary"].items():
            stats_str = ", ".join([f"{k}={v}" for k, v in stats_dict.items()])
            print(f"  {sid}: {stats_str}")

        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)
        saved_dir = save_fold_artifacts(result, cfg)

        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))
        print(f"Saved inference artifacts to: {saved_dir}")

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}

    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in pers_metrics.items()}

    print("\n================ FINAL LOSO RESULTS ================")
    print(
        "Global      -> "
        f"ACC: {global_mean['acc']:.4f} ± {global_std['acc']:.4f}, "
        f"Macro-F1: {global_mean['f1']:.4f} ± {global_std['f1']:.4f}, "
        f"ECE: {global_mean['ece']:.4f} ± {global_std['ece']:.4f}, "
        f"Brier: {global_mean['brier']:.4f} ± {global_std['brier']:.4f}"
    )
    print(
        "Personalized-> "
        f"ACC: {pers_mean['acc']:.4f} ± {pers_std['acc']:.4f}, "
        f"Macro-F1: {pers_mean['f1']:.4f} ± {pers_std['f1']:.4f}, "
        f"ECE: {pers_mean['ece']:.4f} ± {pers_std['ece']:.4f}, "
        f"Brier: {pers_mean['brier']:.4f} ± {pers_std['brier']:.4f}"
    )

    save_experiment_summary(
        cfg=cfg,
        all_results=all_results,
        global_mean=global_mean,
        global_std=global_std,
        pers_mean=pers_mean,
        pers_std=pers_std,
        label_map=label_map,
    )
    save_loso_csv_tables(cfg, all_results)

    return {
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }


# Override all-subject preparation so it also supports pre-windowed UCI HAR arrays.
def build_all_subjects_training_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    label_map: Dict[int, int],
    cfg: Config,
) -> FullTrainingData:
    subject_ids = sorted(subject_arrays.keys())
    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        packaged = build_prewindowed_subject_partitions_train(X, y, cfg) if X.ndim == 3 else build_subject_partitions_train(X, y, cfg)
        train_subjects[sid] = packaged
        print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in subject_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in subject_ids:
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for sid in subject_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    first_X = next(iter(subject_arrays.values()))[0]
    feat_dim = first_X.shape[-1] if first_X.ndim == 3 else first_X.shape[1]
    return FullTrainingData(
        subject_ids=subject_ids,
        train_subjects=train_subjects,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
        scaler=scaler,
    )


# Override the original all-subject driver to use the dataset adapter.
def run_all_subjects_training(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_arrays, label_map = load_dataset_subject_arrays(cfg)

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS USED FOR TRAINING ================")
    print(sorted(subject_arrays.keys()))
    print(f"Label map (activity_id -> class_idx): {label_map}")

    data = build_all_subjects_training_data(subject_arrays, label_map, cfg)
    loaders = all_subjects_to_loaders(data, cfg)

    print("Subject summary:")
    for sid, stats_dict in loaders["summary"].items():
        stats_str = ", ".join([f"{k}={v}" for k, v in stats_dict.items()])
        print(f"  {sid}: {stats_str}")

    result = train_all_subjects_model(loaders, cfg, device)
    saved_dir = save_all_subjects_artifacts(result, cfg)

    print("\n================ FINAL ALL-SUBJECTS MODEL ================")
    print(pretty_metric_line("Combined validation", result["combined_val"]))
    print(f"Saved inference artifacts to: {saved_dir}")

    save_json(Path(cfg.save_dir) / "all_subjects_summary.json", {
        "config": asdict(cfg),
        "label_map": {int(k): int(v) for k, v in label_map.items()},
        "idx_to_activity": inverse_label_map(label_map),
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
    })

    return {
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
        "label_map": label_map,
        "artifact_dir": saved_dir,
    }

## Experiment driver

LOSO experiment orchestration, summary generation, artifact saving, and training over folds.

In [ ]:
# ============================================================
# Experiment driver
# ============================================================

def pretty_metric_line(name: str, metrics: Dict[str, float]) -> str:
    return (
        f"{name} -> ACC: {metrics['acc']:.4f}, "
        f"Macro-F1: {metrics['f1']:.4f}, "
        f"ECE: {metrics['ece']:.4f}, "
        f"Brier: {metrics['brier']:.4f}"
    )


def run_loso_experiment(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS ================")
    print(subject_ids)
    print(f"Label map (activity_id -> class_idx): {label_map}")

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)

        print("Fold summary:")
        for sid, stats in loaders["summary"].items():
            stats_str = ", ".join([f"{k}={v}" for k, v in stats.items()])
            print(f"  {sid}: {stats_str}")

        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)
        saved_dir = save_fold_artifacts(result, cfg)

        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))
        print(f"Saved inference artifacts to: {saved_dir}")

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}

    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v)) for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v)) for k, v in pers_metrics.items()}

    print("\n================ FINAL LOSO RESULTS ================")
    print(
        "Global      -> "
        f"ACC: {global_mean['acc']:.4f} ± {global_std['acc']:.4f}, "
        f"Macro-F1: {global_mean['f1']:.4f} ± {global_std['f1']:.4f}, "
        f"ECE: {global_mean['ece']:.4f} ± {global_std['ece']:.4f}, "
        f"Brier: {global_mean['brier']:.4f} ± {global_std['brier']:.4f}"
    )
    print(
        "Personalized-> "
        f"ACC: {pers_mean['acc']:.4f} ± {pers_std['acc']:.4f}, "
        f"Macro-F1: {pers_mean['f1']:.4f} ± {pers_std['f1']:.4f}, "
        f"ECE: {pers_mean['ece']:.4f} ± {pers_std['ece']:.4f}, "
        f"Brier: {pers_mean['brier']:.4f} ± {pers_std['brier']:.4f}"
    )

    gate = result["adapt_gate"]
    print(
        "Adapt gate -> "
        f"used_personalization={gate['used_personalization']} | "
        f"baseline_score={gate['baseline_score']:.4f} | "
        f"personalized_score={gate['personalized_score']:.4f} | "
        f"required_margin={gate['required_margin']:.4f}"
    )

    save_experiment_summary(
        cfg=cfg,
        all_results=all_results,
        global_mean=global_mean,
        global_std=global_std,
        pers_mean=pers_mean,
        pers_std=pers_std,
        label_map=label_map,
    )

    return {
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }


def build_all_subjects_training_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    label_map: Dict[int, int],
    cfg: Config,
) -> FullTrainingData:
    subject_ids = sorted(subject_arrays.keys())
    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        packaged = build_subject_partitions_train(X, y, cfg)
        train_subjects[sid] = packaged
        print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in subject_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in subject_ids:
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for sid in subject_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    feat_dim = next(iter(subject_arrays.values()))[0].shape[1]
    return FullTrainingData(
        subject_ids=subject_ids,
        train_subjects=train_subjects,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
        scaler=scaler,
    )


def build_all_subjects_summary(data: FullTrainingData) -> Dict[str, Dict[str, int]]:
    out: Dict[str, Dict[str, int]] = {}
    for sid in data.subject_ids:
        out[sid] = {
            "train": int(len(data.train_subjects[sid]["train"].labels)),
            "val": int(len(data.train_subjects[sid]["val"].labels)),
        }
    return out


def all_subjects_to_loaders(
    data: FullTrainingData,
    cfg: Config,
) -> Dict[str, object]:
    loaders: Dict[str, object] = {
        "train_clients": {},
        "num_classes": data.num_classes,
        "feature_dim": data.feature_dim,
        "subject_ids": data.subject_ids,
        "label_map": data.label_map,
        "scaler": data.scaler,
        "summary": build_all_subjects_summary(data),
    }

    for sid in data.subject_ids:
        loaders["train_clients"][sid] = {
            "train": make_loader(data.train_subjects[sid]["train"], cfg.batch_size, True, cfg),
            "val": make_loader(data.train_subjects[sid]["val"], cfg.batch_size, False, cfg),
            "n_train": len(data.train_subjects[sid]["train"].labels),
            "n_val": len(data.train_subjects[sid]["val"].labels),
        }

    return loaders


def combine_partitions(parts: List[SubjectPartition], cfg: Config) -> SubjectPartition:
    feat_dim = parts[0].windows.shape[-1] if parts else 0
    return concat_partitions(parts, cfg.seq_len, feat_dim)


@torch.inference_mode()
def evaluate_combined_validation(
    model: nn.Module,
    train_clients: Dict[str, Dict[str, Any]],
    client_A: Dict[str, torch.Tensor],
    client_log_tau: Dict[str, torch.Tensor],
    global_proto: torch.Tensor,
    proto_basis: torch.Tensor,
    cfg: Config,
    device: torch.device,
) -> Dict[str, float]:
    all_probs: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    for sid, client_info in train_clients.items():
        probs, labels = predict_loader(
            model=model,
            loader=client_info["val"],
            global_proto=global_proto,
            proto_basis=proto_basis,
            client_A=client_A[sid].to(device),
            log_tau=client_log_tau[sid].to(device),
            device=device,
            cfg=cfg,
            use_amp=cfg.amp,
        )
        if len(labels) > 0:
            all_probs.append(probs)
            all_labels.append(labels)

    if not all_probs:
        raise RuntimeError("No validation predictions were produced.")

    probs_cat = np.concatenate(all_probs, axis=0)
    labels_cat = np.concatenate(all_labels, axis=0)
    return summarize_probs(probs_cat, labels_cat, num_classes=global_proto.shape[0], ece_bins=cfg.ece_bins)


def train_all_subjects_model(
    loaders: Dict[str, object],
    cfg: Config,
    device: torch.device,
) -> Dict[str, object]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    subject_ids = loaders["subject_ids"]
    train_clients = loaders["train_clients"]

    global_model = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    proto_basis = orthonormal_random(cfg.emb_dim, cfg.proto_rank, device)

    client_A = {sid: torch.zeros(num_classes, cfg.proto_rank, dtype=torch.float32) for sid in subject_ids}
    client_log_tau = {
        sid: torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32))
        for sid in subject_ids
    }

    best_snapshot = None
    best_score = -1.0
    history: List[Dict[str, float]] = []

    warmup_rounds = min(3, max(1, cfg.rounds // 4))
    round_bar = tqdm(range(1, cfg.rounds + 1), desc="All-subject training", leave=True)

    for rnd in round_bar:
        local_state_dicts = []
        local_weights = []
        proto_summaries: Dict[str, Dict[str, torch.Tensor]] = {}

        for sid in subject_ids:
            state_dict, A_local, log_tau_local, proto_summary = train_one_client(
                global_model=global_model,
                train_loader=train_clients[sid]["train"],
                val_loader=train_clients[sid]["val"],
                global_proto=global_proto,
                proto_basis=proto_basis,
                A_init=client_A[sid],
                log_tau_init=client_log_tau[sid],
                cfg=cfg,
                device=device,
                warmup_extract=(rnd <= warmup_rounds),
            )

            local_state_dicts.append(state_dict)
            local_weights.append(train_clients[sid]["n_train"])
            proto_summaries[sid] = proto_summary
            client_log_tau[sid] = log_tau_local.clone()

        avg_state = average_state_dicts(local_state_dicts, local_weights)
        global_model.load_state_dict(avg_state)

        global_proto, proto_basis, new_client_A = server_geometry_update(
            global_proto=global_proto,
            proto_basis=proto_basis,
            proto_summaries=proto_summaries,
            cfg=cfg,
            device=device,
        )
        client_A = new_client_A

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        per_client_val_sizes: Dict[str, int] = {}

        with torch.no_grad():
            for sid in subject_ids:
                metrics = evaluate_loader(
                    global_model,
                    train_clients[sid]["val"],
                    global_proto,
                    proto_basis,
                    client_A[sid].to(device),
                    client_log_tau[sid].to(device),
                    cfg,
                    device,
                )
                per_client_metrics[sid] = metrics
                per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)

        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(
            per_client_metrics, per_client_val_sizes, cfg
        )
        combined_val = evaluate_combined_validation(
            model=global_model,
            train_clients=train_clients,
            client_A=client_A,
            client_log_tau=client_log_tau,
            global_proto=global_proto,
            proto_basis=proto_basis,
            cfg=cfg,
            device=device,
        )

        round_bar.set_postfix({
            "val_f1": f"{mean_f1:.4f}",
            "val_acc": f"{mean_acc:.4f}",
            "val_ece": f"{mean_ece:.4f}",
        })

        history.append({
            "round": rnd,
            "mean_val_f1": mean_f1,
            "mean_val_acc": mean_acc,
            "mean_val_ece": mean_ece,
            "combined_val_f1": combined_val["f1"],
            "combined_val_acc": combined_val["acc"],
            "combined_val_ece": combined_val["ece"],
            "combined_val_brier": combined_val["brier"],
        })

        if score > best_score:
            best_score = score
            best_snapshot = {
                "model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()},
                "global_proto": global_proto.detach().cpu().clone(),
                "proto_basis": proto_basis.detach().cpu().clone(),
                "client_A": {sid: a.detach().cpu().clone() for sid, a in client_A.items()},
                "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in client_log_tau.items()},
                "per_client_val_metrics": copy.deepcopy(per_client_metrics),
                "combined_val": copy.deepcopy(combined_val),
                "history": copy.deepcopy(history),
            }

    assert best_snapshot is not None

    saved_log_tau = best_snapshot["client_log_tau"]
    train_size_weights = torch.tensor(
        [train_clients[sid]["n_train"] for sid in subject_ids],
        dtype=torch.float32,
        device=device,
    )
    tau_stack = torch.stack([saved_log_tau[sid].float().to(device) for sid in subject_ids], dim=0)
    mean_log_tau = (train_size_weights * tau_stack).sum() / train_size_weights.sum()

    zero_A = torch.zeros(num_classes, cfg.proto_rank, dtype=torch.float32)

    return {
        "run_name": "all_subjects",
        "summary": loaders["summary"],
        "history": best_snapshot["history"],
        "combined_val": best_snapshot["combined_val"],
        "per_client_val_metrics": best_snapshot["per_client_val_metrics"],
        "artifact_bundle": {
            "run_name": "all_subjects",
            "feature_dim": feature_dim,
            "num_classes": num_classes,
            "config": asdict(cfg),
            "model_state": {k: v.detach().cpu().clone() for k, v in best_snapshot["model_state"].items()},
            "global_proto": best_snapshot["global_proto"].detach().cpu().clone(),
            "proto_basis": best_snapshot["proto_basis"].detach().cpu().clone(),
            "mean_log_tau": mean_log_tau.detach().cpu().clone(),
            "selected_A": zero_A.detach().cpu().clone(),
            "selected_log_tau": mean_log_tau.detach().cpu().clone(),
            "default_prediction_head": "global",
            "client_A": {sid: a.detach().cpu().clone() for sid, a in best_snapshot["client_A"].items()},
            "client_log_tau": {sid: t.detach().cpu().clone() for sid, t in best_snapshot["client_log_tau"].items()},
            "label_map": {int(k): int(v) for k, v in loaders["label_map"].items()},
            "idx_to_activity": inverse_label_map(loaders["label_map"]),
            "feature_columns": dataset_feature_columns(cfg),
            "scaler_state": scaler_to_state(loaders["scaler"]),
        },
    }


def save_all_subjects_artifacts(result: Dict[str, Any], cfg: Config) -> str:
    out_dir = Path(cfg.save_dir) / "all_subjects_model"
    out_dir.mkdir(parents=True, exist_ok=True)

    torch.save(result["artifact_bundle"], out_dir / "inference_bundle.pt")
    save_json(out_dir / "metrics.json", {
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
    })
    save_json(out_dir / "history.json", {"history": result["history"]})
    save_json(out_dir / "subject_summary.json", result["summary"])
    pd.DataFrame(result["history"]).to_csv(out_dir / "training_history.csv", index=False)
    pd.DataFrame.from_dict(result["per_client_val_metrics"], orient="index").reset_index().rename(columns={"index": "subject_id"}).to_csv(out_dir / "per_subject_val_metrics.csv", index=False)
    return str(out_dir)


def run_all_subjects_training(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_files = discover_subject_files(cfg)
    subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
    label_map = build_global_label_map(subject_dfs)
    subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS USED FOR TRAINING ================")
    print(sorted(subject_arrays.keys()))
    print(f"Label map (activity_id -> class_idx): {label_map}")

    data = build_all_subjects_training_data(subject_arrays, label_map, cfg)
    loaders = all_subjects_to_loaders(data, cfg)

    print("Subject summary:")
    for sid, stats in loaders["summary"].items():
        stats_str = ", ".join([f"{k}={v}" for k, v in stats.items()])
        print(f"  {sid}: {stats_str}")

    result = train_all_subjects_model(loaders, cfg, device)
    saved_dir = save_all_subjects_artifacts(result, cfg)

    print("\n================ FINAL ALL-SUBJECTS MODEL ================")
    print(pretty_metric_line("Combined validation", result["combined_val"]))
    print(f"Saved inference artifacts to: {saved_dir}")

    save_json(Path(cfg.save_dir) / "all_subjects_summary.json", {
        "config": asdict(cfg),
        "label_map": {int(k): int(v) for k, v in label_map.items()},
        "idx_to_activity": inverse_label_map(label_map),
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
    })

    return {
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
        "label_map": label_map,
        "artifact_dir": saved_dir,
    }

## Inference usage

Example comments showing how to load an inference bundle and predict from new sensor data.

In [ ]:
# ============================================================
# Inference usage
# ============================================================

# Example after training:
# bundle_path = "/content/drive/MyDrive/ccd_pamap2_runs/all_subjects_model/inference_bundle.pt"
# sensor_df = pd.read_csv("/content/your_sensor_file.csv")
# pred_df = predict_sensor_dataframe(sensor_df, bundle_path=bundle_path, device="cuda")
# print(pred_df.head())

## README-aligned dataset adapter: PAMAP2 or UCI HAR

UCI HAR raw inertial-signal loader aligned with the dataset README, including official window length and channel layout.

In [ ]:
# ============================================================
# README-aligned dataset adapter: PAMAP2 or UCI HAR
# ============================================================

UCI_HAR_SIGNAL_FILES: Tuple[str, ...] = (
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
)

UCI_HAR_FEATURE_NAMES: Tuple[str, ...] = UCI_HAR_SIGNAL_FILES

# Official UCI HAR labels from activity_labels.txt / README.
UCI_HAR_ACTIVITY_NAMES: Dict[int, str] = {
    1: "WALKING",
    2: "WALKING_UPSTAIRS",
    3: "WALKING_DOWNSTAIRS",
    4: "SITTING",
    5: "STANDING",
    6: "LAYING",
}

UCI_HAR_EXPECTED_SEQ_LEN = 128          # 2.56 s × 50 Hz
UCI_HAR_EXPECTED_FEATURE_DIM = 9        # total_acc(3) + body_acc(3) + body_gyro(3)
UCI_HAR_SAMPLING_HZ = 50
UCI_HAR_OFFICIAL_OVERLAP = 0.50


def dataset_feature_columns(cfg: Config) -> List[str]:
    if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har":
        return list(UCI_HAR_FEATURE_NAMES)
    return feature_columns()


def find_uci_har_root(data_dir: Union[str, Path]) -> Path:
    """
    Return the official UCI HAR Dataset root folder.

    Expected README structure:
      root/
        README.txt
        activity_labels.txt
        features.txt
        train/X_train.txt
        train/y_train.txt
        train/subject_train.txt
        train/Inertial Signals/*.txt
        test/X_test.txt
        test/y_test.txt
        test/subject_test.txt
        test/Inertial Signals/*.txt
    """
    root = Path(data_dir)

    def is_uci_root(p: Path) -> bool:
        return (
            (p / "train" / "Inertial Signals").exists()
            and (p / "test" / "Inertial Signals").exists()
            and (p / "train" / "y_train.txt").exists()
            and (p / "test" / "y_test.txt").exists()
            and (p / "train" / "subject_train.txt").exists()
            and (p / "test" / "subject_test.txt").exists()
        )

    if is_uci_root(root):
        return root

    # Common after unzipping: /content/drive/MyDrive/.../UCI HAR Dataset/UCI HAR Dataset
    nested = root / "UCI HAR Dataset"
    if is_uci_root(nested):
        return nested

    # Fallback search under the supplied folder first, then MyDrive.
    search_roots = [root, Path("/content/drive/MyDrive")]
    seen: set[str] = set()
    for base in search_roots:
        if not base.exists() or str(base) in seen:
            continue
        seen.add(str(base))
        for p in base.rglob("Inertial Signals"):
            candidate = p.parent.parent
            if is_uci_root(candidate):
                return candidate

    raise FileNotFoundError(
        "Could not find the official UCI HAR Dataset folder. Set cfg.data_dir to the folder containing "
        "train/Inertial Signals, test/Inertial Signals, y_*.txt, and subject_*.txt. "
        f"Current data_dir={str(data_dir)!r}"
    )


def read_uci_activity_labels(root: Path) -> Dict[int, str]:
    """Read activity_labels.txt when available; otherwise use the README labels."""
    labels_path = root / "activity_labels.txt"
    if not labels_path.exists():
        return dict(UCI_HAR_ACTIVITY_NAMES)

    labels: Dict[int, str] = {}
    with open(labels_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                labels[int(parts[0])] = parts[1]
    return labels if labels else dict(UCI_HAR_ACTIVITY_NAMES)


def validate_uci_har_root(root: Path) -> Dict[str, Any]:
    """Validate files described in README and return dataset metadata."""
    activity_names = read_uci_activity_labels(root)

    missing: List[str] = []
    expected_common = ["README.txt", "features_info.txt", "features.txt", "activity_labels.txt"]
    for fname in expected_common:
        if not (root / fname).exists():
            missing.append(fname)

    for split in ["train", "test"]:
        suffix = split
        split_dir = root / split
        for fname in [f"X_{suffix}.txt", f"y_{suffix}.txt", f"subject_{suffix}.txt"]:
            if not (split_dir / fname).exists():
                missing.append(f"{split}/{fname}")
        inertial_dir = split_dir / "Inertial Signals"
        for sig in UCI_HAR_SIGNAL_FILES:
            fname = f"{sig}_{suffix}.txt"
            if not (inertial_dir / fname).exists():
                missing.append(f"{split}/Inertial Signals/{fname}")

    if missing:
        raise FileNotFoundError(
            "The UCI HAR folder was found, but these README-listed files are missing:\n  - "
            + "\n  - ".join(missing)
        )

    return {
        "dataset": "UCI HAR Dataset",
        "sampling_hz": UCI_HAR_SAMPLING_HZ,
        "window_seconds": UCI_HAR_EXPECTED_SEQ_LEN / UCI_HAR_SAMPLING_HZ,
        "readings_per_window": UCI_HAR_EXPECTED_SEQ_LEN,
        "official_overlap": UCI_HAR_OFFICIAL_OVERLAP,
        "raw_inertial_channels": list(UCI_HAR_SIGNAL_FILES),
        "num_raw_inertial_channels": UCI_HAR_EXPECTED_FEATURE_DIM,
        "activity_names": activity_names,
        "note": "Uses raw Inertial Signals files, not the 561-feature X_train/X_test vectors.",
    }


def load_uci_split(root: Path, split: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load one official UCI HAR split.

    Returns:
      X        : [N, 128, 9] raw inertial windows
      y        : [N] official labels in {1,...,6}
      subjects : [N] official subject IDs in {1,...,30}
    """
    if split not in {"train", "test"}:
        raise ValueError("split must be 'train' or 'test'")

    split_dir = root / split
    inertial_dir = split_dir / "Inertial Signals"
    suffix = split

    signals: List[np.ndarray] = []
    for sig in UCI_HAR_SIGNAL_FILES:
        fp = inertial_dir / f"{sig}_{suffix}.txt"
        arr = np.loadtxt(fp, dtype=np.float32)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        if arr.shape[1] != UCI_HAR_EXPECTED_SEQ_LEN:
            raise RuntimeError(
                f"Unexpected window length in {fp}: got {arr.shape[1]}, "
                f"expected {UCI_HAR_EXPECTED_SEQ_LEN} according to the README."
            )
        signals.append(arr)

    X = np.stack(signals, axis=-1).astype(np.float32)  # [N, 128, 9]
    y = np.loadtxt(split_dir / f"y_{suffix}.txt", dtype=np.int64).reshape(-1)
    subjects = np.loadtxt(split_dir / f"subject_{suffix}.txt", dtype=np.int64).reshape(-1)

    if X.shape[0] != len(y) or len(y) != len(subjects):
        raise RuntimeError(
            f"UCI HAR {split} length mismatch: X={X.shape[0]}, y={len(y)}, subjects={len(subjects)}"
        )

    if X.shape[1:] != (UCI_HAR_EXPECTED_SEQ_LEN, UCI_HAR_EXPECTED_FEATURE_DIM):
        raise RuntimeError(f"Unexpected UCI HAR tensor shape for {split}: {X.shape}")

    labels = set(np.unique(y).tolist())
    if not labels.issubset(set(UCI_HAR_ACTIVITY_NAMES.keys())):
        raise RuntimeError(f"Unexpected UCI HAR labels in {split}: {sorted(labels)}")

    return X, y, subjects


def load_uci_har_subject_arrays(cfg: Config) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray]], Dict[int, int]]:
    """
    Load README-defined UCI HAR raw inertial windows and group them by subject.

    Important protocol choice for Review 2:
      The official train/test split is ignored for final LOSO, because the reviewer asked for
      subject-level leave-one-subject-out evaluation. We therefore combine train and test files,
      group all windows by subject ID, and then choose one subject as held-out in each fold.
    """
    root = find_uci_har_root(cfg.data_dir)
    meta = validate_uci_har_root(root)

    print(f"[UCI HAR] Using dataset root: {root}")
    print(
        "[UCI HAR] README-aligned raw input: "
        f"{meta['readings_per_window']} readings/window, "
        f"{meta['window_seconds']:.2f}s, {meta['sampling_hz']}Hz, "
        f"{meta['num_raw_inertial_channels']} inertial channels, "
        f"{int(meta['official_overlap'] * 100)}% official overlap."
    )
    print("[UCI HAR] The 561-feature X_train/X_test vectors are present but are not used by CDPL.")

    X_train, y_train, s_train = load_uci_split(root, "train")
    X_test, y_test, s_test = load_uci_split(root, "test")

    X_all = np.concatenate([X_train, X_test], axis=0).astype(np.float32)
    y_all_orig = np.concatenate([y_train, y_test], axis=0).astype(np.int64)
    s_all = np.concatenate([s_train, s_test], axis=0).astype(np.int64)

    # Fixed official map: labels 1..6 become class indices 0..5.
    label_map: Dict[int, int] = {activity_id: activity_id - 1 for activity_id in sorted(UCI_HAR_ACTIVITY_NAMES)}
    y_all = np.asarray([label_map[int(a)] for a in y_all_orig], dtype=np.int64)

    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for sid_num in sorted(np.unique(s_all).tolist()):
        if sid_num < 1 or sid_num > 30:
            raise RuntimeError(f"Unexpected UCI HAR subject ID: {sid_num}")
        mask = s_all == sid_num
        sid = f"subject{int(sid_num):02d}"
        subject_arrays[sid] = (X_all[mask].astype(np.float32), y_all[mask].astype(np.int64))

    if len(subject_arrays) != 30:
        print(f"[WARN] README says 30 volunteers, but loaded {len(subject_arrays)} subjects.")

    print(
        f"[UCI HAR] Loaded {len(subject_arrays)} subjects | "
        f"windows={len(y_all)} | X={tuple(X_all.shape)} | "
        f"class indices 0..5 -> {meta['activity_names']}"
    )
    return subject_arrays, label_map


def load_dataset_subject_arrays(cfg: Config) -> Tuple[Dict[str, Tuple[np.ndarray, np.ndarray]], Dict[int, int]]:
    dataset = getattr(cfg, "dataset_name", "pamap2").lower()
    if dataset == "uci_har":
        return load_uci_har_subject_arrays(cfg)
    if dataset == "pamap2":
        subject_files = discover_subject_files(cfg)
        subject_dfs = {sid: load_subject_dataframe(path, cfg) for sid, path in subject_files.items()}
        label_map = build_global_label_map(subject_dfs)
        subject_arrays = {sid: encode_subject(df, label_map) for sid, df in subject_dfs.items()}
        return subject_arrays, label_map
    raise ValueError(f"Unknown dataset_name={cfg.dataset_name!r}. Use 'pamap2' or 'uci_har'.")


def split_counts_with_gap(total: int, ratios: List[float], min_counts: List[int], gap: int) -> Optional[List[Tuple[int, int]]]:
    """Split index positions [0,total) into contiguous ranges with purge gaps between them."""
    k = len(ratios)
    usable = total - gap * (k - 1)
    if usable < sum(min_counts):
        return None

    lengths = allocate_lengths_with_minimum(usable, ratios, min_counts)
    if lengths is None:
        return None

    ranges: List[Tuple[int, int]] = []
    pos = 0
    for j, length in enumerate(lengths):
        ranges.append((pos, pos + length))
        pos += length
        if j < k - 1:
            pos += gap
    return ranges


def prewindowed_split_by_class(
    X: np.ndarray,
    y: np.ndarray,
    split_names: List[str],
    ratios: List[float],
    cfg: Config,
    train_mode: bool,
) -> Dict[str, SubjectPartition]:
    """
    Split already-windowed UCI HAR data by class while preserving local file order.

    UCI HAR windows are already fixed-width 2.56 s windows with 50% overlap. Therefore this
    function does not perform another sliding-window step. It only divides the existing windows
    into train/val or support/adapt_val/test splits. A gap of one pre-windowed sample is used by
    default so adjacent 50%-overlapping windows do not fall into different sub-splits.
    """
    assert X.ndim == 3, f"Expected UCI HAR prewindowed X with shape [N,T,F], got {X.shape}"
    if X.shape[1] != UCI_HAR_EXPECTED_SEQ_LEN or X.shape[2] != UCI_HAR_EXPECTED_FEATURE_DIM:
        raise RuntimeError(f"Unexpected UCI HAR prewindowed tensor shape: {X.shape}")

    feat_dim = X.shape[-1]
    out: Dict[str, List[SubjectPartition]] = {name: [] for name in split_names}
    gap = int(getattr(cfg, "uci_purge_gap_windows", 1))

    classes = sorted(np.unique(y).tolist())
    for cls in classes:
        idx = np.where(y == cls)[0]
        n = len(idx)

        if train_mode:
            if n < 2:
                out["train"].append(SubjectPartition(X[idx], y[idx]))
                continue
            min_counts = [1, 1]
        else:
            if n < 3:
                out["test"].append(SubjectPartition(X[idx], y[idx]))
                continue
            min_counts = [1, 1, 1]

        ranges = split_counts_with_gap(n, ratios, min_counts=min_counts, gap=gap)
        if ranges is None:
            ranges = split_counts_with_gap(n, ratios, min_counts=min_counts, gap=0)
        if ranges is None:
            target = "train" if train_mode else "test"
            out[target].append(SubjectPartition(X[idx], y[idx]))
            continue

        for name, (a, b) in zip(split_names, ranges):
            chosen = idx[a:b]
            if len(chosen) > 0:
                out[name].append(SubjectPartition(X[chosen], y[chosen]))

    return {name: concat_partitions(parts, cfg.seq_len, feat_dim) for name, parts in out.items()}


def build_prewindowed_subject_partitions_train(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    return prewindowed_split_by_class(
        X=X,
        y=y,
        split_names=["train", "val"],
        ratios=[cfg.train_subject_train_ratio, cfg.train_subject_val_ratio],
        cfg=cfg,
        train_mode=True,
    )


def build_prewindowed_subject_partitions_heldout(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
) -> Dict[str, SubjectPartition]:
    return prewindowed_split_by_class(
        X=X,
        y=y,
        split_names=["support", "adapt_val", "test"],
        ratios=[cfg.heldout_support_ratio, cfg.heldout_adapt_val_ratio, cfg.heldout_test_ratio],
        cfg=cfg,
        train_mode=False,
    )


# Override the original PAMAP2-only build_fold_data so it also accepts UCI HAR [N,128,9] windows.
def build_fold_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    heldout_id: str,
    label_map: Dict[int, int],
    cfg: Config,
) -> FoldData:
    train_ids = [sid for sid in subject_arrays.keys() if sid != heldout_id]

    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    heldout_subject: Dict[str, SubjectPartition] = {}
    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        if X.ndim == 3:
            packaged = (
                build_prewindowed_subject_partitions_heldout(X, y, cfg)
                if sid == heldout_id
                else build_prewindowed_subject_partitions_train(X, y, cfg)
            )
        else:
            packaged = (
                build_subject_partitions_heldout(X, y, cfg)
                if sid == heldout_id
                else build_subject_partitions_train(X, y, cfg)
            )

        if sid == heldout_id:
            heldout_subject = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=True)
        else:
            train_subjects[sid] = packaged
            print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in train_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found for scaler fitting.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in train_subjects.keys():
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for split_name in heldout_subject.keys():
        heldout_subject[split_name] = transform_partition(heldout_subject[split_name])

    for sid in train_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    for split_name in ("support", "adapt_val", "test"):
        if len(heldout_subject[split_name].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(
                f"Held-out {heldout_id} split {split_name} has too few windows "
                f"({len(heldout_subject[split_name].windows)})."
            )

    first_X = next(iter(subject_arrays.values()))[0]
    feat_dim = first_X.shape[-1] if first_X.ndim == 3 else first_X.shape[1]
    return FoldData(
        heldout_id=heldout_id,
        train_ids=train_ids,
        train_subjects=train_subjects,
        heldout_subject=heldout_subject,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
        scaler=scaler,
    )


def metric_mean_std_ci(values: List[float]) -> Dict[str, float]:
    vals = np.asarray(values, dtype=np.float64)
    n = len(vals)
    mean = float(np.mean(vals)) if n > 0 else float("nan")
    std = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    if n > 1:
        try:
            from scipy import stats
            tcrit = float(stats.t.ppf(0.975, n - 1))
        except Exception:
            tcrit = 1.96
        half = float(tcrit * std / math.sqrt(n))
    else:
        half = 0.0
    return {
        "n": int(n),
        "mean": mean,
        "std": std,
        "ci95_low": mean - half,
        "ci95_high": mean + half,
    }


def save_loso_csv_tables(cfg: Config, all_results: List[Dict[str, Any]]) -> None:
    """Save per-subject results and paper-ready mean/std/95% CI tables."""
    out_dir = Path(cfg.save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    fold_rows: List[Dict[str, Any]] = []
    for res in all_results:
        row: Dict[str, Any] = {"dataset": cfg.dataset_name, "heldout_id": res["heldout_id"]}
        for prefix, key in [("global", "global_test"), ("personalized", "personalized_test")]:
            for m, v in res[key].items():
                row[f"{prefix}_{m}"] = float(v)
        row["used_personalization"] = bool(res["adapt_gate"]["used_personalization"])
        row["baseline_score"] = float(res["adapt_gate"]["baseline_score"])
        row["personalized_score"] = float(res["adapt_gate"]["personalized_score"])
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(out_dir / f"{cfg.dataset_name}_cdpl_fold_results.csv", index=False)

    summary_rows: List[Dict[str, Any]] = []
    for variant, prefix in [("CDPL Global", "global"), ("CDPL Personalized", "personalized")]:
        for metric in ["acc", "f1", "ece", "brier"]:
            stats_dict = metric_mean_std_ci(fold_df[f"{prefix}_{metric}"].tolist())
            summary_rows.append({
                "dataset": cfg.dataset_name,
                "method": variant,
                "metric": metric,
                **stats_dict,
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / f"{cfg.dataset_name}_cdpl_summary_ci.csv", index=False)

    # Wide table for direct paper use.
    wide_rows: List[Dict[str, Any]] = []
    for variant, prefix in [("CDPL Global", "global"), ("CDPL Personalized", "personalized")]:
        row = {"dataset": cfg.dataset_name, "method": variant}
        for metric in ["acc", "f1", "ece", "brier"]:
            s = metric_mean_std_ci(fold_df[f"{prefix}_{metric}"].tolist())
            row[metric] = f"{s['mean']:.4f} ± {s['std']:.4f} [{s['ci95_low']:.4f}, {s['ci95_high']:.4f}]"
        wide_rows.append(row)
    pd.DataFrame(wide_rows).to_csv(out_dir / f"{cfg.dataset_name}_cdpl_paper_table.csv", index=False)


# Override the original run_loso_experiment to load either PAMAP2 or UCI HAR.
def run_loso_experiment(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_arrays, label_map = load_dataset_subject_arrays(cfg)

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS ================")
    print(subject_ids)
    print(f"Label map (official activity_id -> class_idx): {label_map}")
    if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har":
        print(f"Activity names: {UCI_HAR_ACTIVITY_NAMES}")
        print("Input tensor per example: [128 time steps, 9 inertial channels]")

    all_results = []
    for heldout_id in subject_ids:
        print(f"\n{'='*18} HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)

        print("Fold summary:")
        for sid, stats_dict in loaders["summary"].items():
            stats_str = ", ".join([f"{k}={v}" for k, v in stats_dict.items()])
            print(f"  {sid}: {stats_str}")

        result = train_one_fold(loaders, cfg, device)
        all_results.append(result)
        saved_dir = save_fold_artifacts(result, cfg)

        print(pretty_metric_line("Global", result["global_test"]))
        print(pretty_metric_line("Personalized", result["personalized_test"]))
        print(f"Saved inference artifacts to: {saved_dir}")

    global_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}
    pers_metrics = {k: [] for k in ("acc", "f1", "ece", "brier")}

    for res in all_results:
        for k in global_metrics.keys():
            global_metrics[k].append(res["global_test"][k])
            pers_metrics[k].append(res["personalized_test"][k])

    global_mean = {k: float(np.mean(v)) for k, v in global_metrics.items()}
    pers_mean = {k: float(np.mean(v)) for k, v in pers_metrics.items()}
    global_std = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in global_metrics.items()}
    pers_std = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in pers_metrics.items()}

    print("\n================ FINAL LOSO RESULTS ================")
    print(
        "Global      -> "
        f"ACC: {global_mean['acc']:.4f} ± {global_std['acc']:.4f}, "
        f"Macro-F1: {global_mean['f1']:.4f} ± {global_std['f1']:.4f}, "
        f"ECE: {global_mean['ece']:.4f} ± {global_std['ece']:.4f}, "
        f"Brier: {global_mean['brier']:.4f} ± {global_std['brier']:.4f}"
    )
    print(
        "Personalized-> "
        f"ACC: {pers_mean['acc']:.4f} ± {pers_std['acc']:.4f}, "
        f"Macro-F1: {pers_mean['f1']:.4f} ± {pers_std['f1']:.4f}, "
        f"ECE: {pers_mean['ece']:.4f} ± {pers_std['ece']:.4f}, "
        f"Brier: {pers_mean['brier']:.4f} ± {pers_std['brier']:.4f}"
    )

    save_experiment_summary(
        cfg=cfg,
        all_results=all_results,
        global_mean=global_mean,
        global_std=global_std,
        pers_mean=pers_mean,
        pers_std=pers_std,
        label_map=label_map,
    )
    save_loso_csv_tables(cfg, all_results)

    return {
        "folds": all_results,
        "global_mean": global_mean,
        "global_std": global_std,
        "personalized_mean": pers_mean,
        "personalized_std": pers_std,
        "label_map": label_map,
    }


# Override all-subject preparation so it also supports pre-windowed UCI HAR arrays.
def build_all_subjects_training_data(
    subject_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    label_map: Dict[int, int],
    cfg: Config,
) -> FullTrainingData:
    subject_ids = sorted(subject_arrays.keys())
    train_subjects: Dict[str, Dict[str, SubjectPartition]] = {}
    num_classes = len(label_map)

    for sid, (X, y) in subject_arrays.items():
        packaged = build_prewindowed_subject_partitions_train(X, y, cfg) if X.ndim == 3 else build_subject_partitions_train(X, y, cfg)
        train_subjects[sid] = packaged
        print_partition_debug(sid, packaged, num_classes, heldout=False)

    scaler = StandardScaler()
    any_train = False
    for sid in subject_ids:
        w = train_subjects[sid]["train"].windows
        if len(w) > 0:
            scaler.partial_fit(w.reshape(-1, w.shape[-1]))
            any_train = True
    if not any_train:
        raise RuntimeError("No training windows found across subjects.")

    def transform_partition(part: SubjectPartition) -> SubjectPartition:
        if len(part.windows) == 0:
            return part
        shape = part.windows.shape
        flat = part.windows.reshape(-1, shape[-1])
        flat = scaler.transform(flat)
        return SubjectPartition(
            windows=flat.reshape(shape).astype(np.float32),
            labels=part.labels.astype(np.int64),
        )

    for sid in subject_ids:
        for split_name in train_subjects[sid].keys():
            train_subjects[sid][split_name] = transform_partition(train_subjects[sid][split_name])

    for sid in subject_ids:
        if len(train_subjects[sid]["train"].windows) < cfg.min_windows_per_partition:
            raise RuntimeError(f"{sid} has too few train windows ({len(train_subjects[sid]['train'].windows)}).")
        if len(train_subjects[sid]["val"].windows) < max(1, cfg.min_windows_per_partition // 2):
            raise RuntimeError(f"{sid} has too few val windows ({len(train_subjects[sid]['val'].windows)}).")

    first_X = next(iter(subject_arrays.values()))[0]
    feat_dim = first_X.shape[-1] if first_X.ndim == 3 else first_X.shape[1]
    return FullTrainingData(
        subject_ids=subject_ids,
        train_subjects=train_subjects,
        num_classes=num_classes,
        feature_dim=feat_dim,
        label_map=label_map,
        scaler=scaler,
    )


# Override the original all-subject driver to use the dataset adapter.
def run_all_subjects_training(cfg: Config) -> Dict[str, object]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    subject_arrays, label_map = load_dataset_subject_arrays(cfg)

    print("\n================ CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")

    print("\n================ SUBJECTS USED FOR TRAINING ================")
    print(sorted(subject_arrays.keys()))
    print(f"Label map (official activity_id -> class_idx): {label_map}")
    if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har":
        print(f"Activity names: {UCI_HAR_ACTIVITY_NAMES}")

    data = build_all_subjects_training_data(subject_arrays, label_map, cfg)
    loaders = all_subjects_to_loaders(data, cfg)

    print("Subject summary:")
    for sid, stats_dict in loaders["summary"].items():
        stats_str = ", ".join([f"{k}={v}" for k, v in stats_dict.items()])
        print(f"  {sid}: {stats_str}")

    result = train_all_subjects_model(loaders, cfg, device)
    saved_dir = save_all_subjects_artifacts(result, cfg)

    print("\n================ FINAL ALL-SUBJECTS MODEL ================")
    print(pretty_metric_line("Combined validation", result["combined_val"]))
    print(f"Saved inference artifacts to: {saved_dir}")

    save_json(Path(cfg.save_dir) / "all_subjects_summary.json", {
        "config": asdict(cfg),
        "label_map": {int(k): int(v) for k, v in label_map.items()},
        "idx_to_activity": inverse_label_map(label_map),
        "activity_names": UCI_HAR_ACTIVITY_NAMES if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har" else None,
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
    })

    return {
        "combined_val": result["combined_val"],
        "per_client_val_metrics": result["per_client_val_metrics"],
        "label_map": label_map,
        "artifact_dir": saved_dir,
    }

## Main

Manual execution launcher. In this notebook it is disabled by default using RUN_FULL_BENCHMARK = False so Run All does not accidentally start a full LOSO benchmark.

In [ ]:
# ============================================================
# Main
# ============================================================

## Review 2 baselines: FedAvg, FedProto, FedRep + CDPL

Additional baseline implementations and benchmark runner for reviewer-requested comparisons and statistical outputs.

In [ ]:
# ============================================================
# Review 2 baselines: FedAvg, FedProto, FedRep + CDPL
# ============================================================

class LinearClassifierModel(nn.Module):
    """Temporal encoder with a shared linear classifier head for FedAvg."""
    def __init__(self, in_dim: int, num_classes: int, cfg: Config):
        super().__init__()
        self.encoder = TemporalEncoder(in_dim, cfg)
        self.head = nn.Linear(cfg.emb_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return self.head(z)


def make_linear_head(num_classes: int, cfg: Config, device: torch.device) -> nn.Linear:
    return nn.Linear(cfg.emb_dim, num_classes).to(device)


def make_class_weight_from_labels(labels: np.ndarray, num_classes: int, device: torch.device, cfg: Config) -> torch.Tensor:
    counts = np.bincount(labels.astype(np.int64), minlength=num_classes).astype(np.float32)
    present = counts > 0
    weights = np.ones(num_classes, dtype=np.float32)
    if present.any():
        ref = float(np.median(counts[present]))
        weights[present] = np.power(ref / np.clip(counts[present], 1.0, None), cfg.class_weight_power)
        weights[present] = np.clip(weights[present], 1.0 / cfg.class_weight_max, cfg.class_weight_max)
        weights[present] /= max(float(weights[present].mean()), 1e-8)
    return torch.tensor(weights, dtype=torch.float32, device=device)


@torch.inference_mode()
def predict_classifier_loader(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())
    if not probs_all:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_classifier_model(model: nn.Module, loader: DataLoader, num_classes: int, cfg: Config, device: torch.device) -> Dict[str, float]:
    probs, labels = predict_classifier_loader(model, loader, device, cfg)
    return summarize_probs(probs, labels, num_classes=num_classes, ece_bins=cfg.ece_bins)


def train_classifier_client(
    global_model: nn.Module,
    train_loader: DataLoader,
    num_classes: int,
    cfg: Config,
    device: torch.device,
) -> Dict[str, torch.Tensor]:
    local_model = clone_model(global_model).to(device)
    local_model.train()
    labels = train_loader.dataset.y.detach().cpu().numpy()
    class_weight = make_class_weight_from_labels(labels, num_classes, device, cfg)
    optimizer = torch.optim.AdamW(local_model.parameters(), lr=cfg.lr_encoder, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.local_epochs))
    scaler = torch.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"

    optimizer_steps = 0
    scheduler_steps = 0
    for _epoch in range(cfg.local_epochs):
        local_model.train()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                logits = local_model(xb)
                loss = F.cross_entropy(logits, yb, weight=class_weight, label_smoothing=0.05)
            if amp_enabled:
                scaler.scale(loss).backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), cfg.grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    optimizer_steps += 1
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(local_model.parameters(), cfg.grad_clip)
                optimizer.step()
                optimizer_steps += 1
        if optimizer_steps > scheduler_steps:
            scheduler.step()
            scheduler_steps += 1
    state = {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}
    del local_model
    return state


def evaluate_weighted_client_validation_classifier(
    model: nn.Module,
    train_clients: Dict[str, Dict[str, Any]],
    train_ids: List[str],
    num_classes: int,
    cfg: Config,
    device: torch.device,
) -> Tuple[Dict[str, float], float]:
    per_client_metrics: Dict[str, Dict[str, float]] = {}
    per_client_val_sizes: Dict[str, int] = {}
    for sid in train_ids:
        m = evaluate_classifier_model(model, train_clients[sid]["val"], num_classes, cfg, device)
        per_client_metrics[sid] = m
        per_client_val_sizes[sid] = len(train_clients[sid]["val"].dataset)
    mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, per_client_val_sizes, cfg)
    return {"f1": mean_f1, "acc": mean_acc, "ece": mean_ece}, score


def train_fedavg_fold(loaders: Dict[str, object], cfg: Config, device: torch.device) -> Dict[str, Any]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    global_model = LinearClassifierModel(feature_dim, num_classes, cfg).to(device)
    best_snapshot: Optional[Dict[str, Any]] = None
    best_score = -1e9
    history: List[Dict[str, float]] = []

    round_bar = tqdm(range(1, cfg.rounds + 1), desc=f"FedAvg {heldout_id}", leave=True)
    for rnd in round_bar:
        local_states: List[Dict[str, torch.Tensor]] = []
        local_weights: List[float] = []
        for sid in train_ids:
            state = train_classifier_client(global_model, train_clients[sid]["train"], num_classes, cfg, device)
            local_states.append(state)
            local_weights.append(train_clients[sid]["n_train"])
        global_model.load_state_dict(average_state_dicts(local_states, local_weights))
        val_metrics, score = evaluate_weighted_client_validation_classifier(global_model, train_clients, train_ids, num_classes, cfg, device)
        history.append({"round": rnd, "mean_val_f1": val_metrics["f1"], "mean_val_acc": val_metrics["acc"], "mean_val_ece": val_metrics["ece"]})
        round_bar.set_postfix({"val_f1": f"{val_metrics['f1']:.4f}", "val_acc": f"{val_metrics['acc']:.4f}", "val_ece": f"{val_metrics['ece']:.4f}"})
        if score > best_score:
            best_score = score
            best_snapshot = {"model_state": {k: v.detach().cpu().clone() for k, v in global_model.state_dict().items()}, "history": copy.deepcopy(history)}

    assert best_snapshot is not None
    global_model.load_state_dict(best_snapshot["model_state"])
    test_metrics = evaluate_classifier_model(global_model, heldout["test"], num_classes, cfg, device)
    return {"heldout_id": heldout_id, "method": "FedAvg", "variant": "Global", "test": test_metrics, "history": best_snapshot["history"]}


def encoder_head_probs(
    encoder: nn.Module,
    head: nn.Module,
    loader: DataLoader,
    device: torch.device,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray]:
    encoder.eval(); head.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    with torch.inference_mode():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = encoder(xb)
                logits = head(z)
                probs = torch.softmax(logits, dim=1)
            probs_all.append(probs.detach().cpu().numpy())
            y_all.append(yb.detach().cpu().numpy())
    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_encoder_head(encoder: nn.Module, head: nn.Module, loader: DataLoader, num_classes: int, cfg: Config, device: torch.device) -> Dict[str, float]:
    probs, labels = encoder_head_probs(encoder, head, loader, device, cfg)
    return summarize_probs(probs, labels, num_classes=num_classes, ece_bins=cfg.ece_bins)


def train_fedrep_client(
    global_encoder: nn.Module,
    head_state: Dict[str, torch.Tensor],
    train_loader: DataLoader,
    num_classes: int,
    cfg: Config,
    device: torch.device,
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    local_encoder = clone_model(global_encoder).to(device)
    local_head = make_linear_head(num_classes, cfg, device)
    local_head.load_state_dict(head_state)
    labels = train_loader.dataset.y.detach().cpu().numpy()
    class_weight = make_class_weight_from_labels(labels, num_classes, device, cfg)
    optimizer = torch.optim.AdamW(list(local_encoder.parameters()) + list(local_head.parameters()), lr=cfg.lr_encoder, weight_decay=cfg.weight_decay)
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp_enabled)

    for _epoch in range(cfg.local_epochs):
        local_encoder.train(); local_head.train()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                logits = local_head(local_encoder(xb))
                loss = F.cross_entropy(logits, yb, weight=class_weight, label_smoothing=0.05)
            if amp_enabled:
                scaler.scale(loss).backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(list(local_encoder.parameters()) + list(local_head.parameters()), cfg.grad_clip)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(list(local_encoder.parameters()) + list(local_head.parameters()), cfg.grad_clip)
                optimizer.step()

    enc_state = {k: v.detach().cpu().clone() for k, v in local_encoder.state_dict().items()}
    h_state = {k: v.detach().cpu().clone() for k, v in local_head.state_dict().items()}
    del local_encoder, local_head
    return enc_state, h_state


def adapt_fedrep_head(
    encoder: nn.Module,
    support_loader: DataLoader,
    adapt_val_loader: DataLoader,
    num_classes: int,
    cfg: Config,
    device: torch.device,
) -> Tuple[Dict[str, torch.Tensor], Dict[str, float]]:
    frozen_encoder = clone_model(encoder).to(device)
    frozen_encoder.eval()
    for p in frozen_encoder.parameters():
        p.requires_grad = False
    head = make_linear_head(num_classes, cfg, device)
    labels = support_loader.dataset.y.detach().cpu().numpy()
    class_weight = make_class_weight_from_labels(labels, num_classes, device, cfg)
    optimizer = torch.optim.AdamW(head.parameters(), lr=cfg.lr_personal, weight_decay=cfg.weight_decay)
    best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    best_metrics = evaluate_encoder_head(frozen_encoder, head, adapt_val_loader, num_classes, cfg, device)
    best_score = personalization_selection_score(best_metrics)
    no_improve = 0
    for _epoch in range(cfg.personalization_epochs):
        head.train()
        for xb, yb in support_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            z = frozen_encoder(xb)
            logits = head(z)
            loss = F.cross_entropy(logits, yb, weight=class_weight, label_smoothing=0.02)
            loss.backward()
            if cfg.grad_clip is not None and cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(head.parameters(), cfg.grad_clip)
            optimizer.step()
        m = evaluate_encoder_head(frozen_encoder, head, adapt_val_loader, num_classes, cfg, device)
        s = personalization_selection_score(m)
        if s > best_score:
            best_score = s
            best_metrics = m
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= cfg.personalization_patience:
            break
    return best_state, best_metrics


def train_fedrep_fold(loaders: Dict[str, object], cfg: Config, device: torch.device) -> Dict[str, Any]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    global_encoder = TemporalEncoder(feature_dim, cfg).to(device)
    head_template = make_linear_head(num_classes, cfg, device)
    client_heads = {sid: {k: v.detach().cpu().clone() for k, v in head_template.state_dict().items()} for sid in train_ids}
    best_snapshot: Optional[Dict[str, Any]] = None
    best_score = -1e9
    history: List[Dict[str, float]] = []

    round_bar = tqdm(range(1, cfg.rounds + 1), desc=f"FedRep {heldout_id}", leave=True)
    for rnd in round_bar:
        enc_states: List[Dict[str, torch.Tensor]] = []
        weights: List[float] = []
        for sid in train_ids:
            enc_state, head_state = train_fedrep_client(global_encoder, client_heads[sid], train_clients[sid]["train"], num_classes, cfg, device)
            enc_states.append(enc_state)
            weights.append(train_clients[sid]["n_train"])
            client_heads[sid] = head_state
        global_encoder.load_state_dict(average_state_dicts(enc_states, weights))

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        val_sizes: Dict[str, int] = {}
        for sid in train_ids:
            head = make_linear_head(num_classes, cfg, device)
            head.load_state_dict(client_heads[sid])
            per_client_metrics[sid] = evaluate_encoder_head(global_encoder, head, train_clients[sid]["val"], num_classes, cfg, device)
            val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        round_bar.set_postfix({"val_f1": f"{mean_f1:.4f}", "val_acc": f"{mean_acc:.4f}", "val_ece": f"{mean_ece:.4f}"})
        if score > best_score:
            best_score = score
            best_snapshot = {
                "encoder_state": {k: v.detach().cpu().clone() for k, v in global_encoder.state_dict().items()},
                "client_heads": copy.deepcopy(client_heads),
                "history": copy.deepcopy(history),
            }

    assert best_snapshot is not None
    global_encoder.load_state_dict(best_snapshot["encoder_state"])
    heldout_head_state, adapt_metrics = adapt_fedrep_head(global_encoder, heldout["support"], heldout["adapt_val"], num_classes, cfg, device)
    heldout_head = make_linear_head(num_classes, cfg, device)
    heldout_head.load_state_dict(heldout_head_state)
    test_metrics = evaluate_encoder_head(global_encoder, heldout_head, heldout["test"], num_classes, cfg, device)
    return {"heldout_id": heldout_id, "method": "FedRep", "variant": "Personalized", "test": test_metrics, "adapt_val": adapt_metrics, "history": best_snapshot["history"]}


def proto_only_logits(z: torch.Tensor, prototypes: torch.Tensor, log_tau: torch.Tensor, cfg: Config) -> torch.Tensor:
    tau = torch.exp(log_tau).clamp(cfg.tau_min, cfg.tau_max)
    return tau * (z @ F.normalize(prototypes, dim=1).T)


def fedproto_loss(
    z: torch.Tensor,
    y: torch.Tensor,
    prototypes: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    class_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    P = F.normalize(prototypes, dim=1)
    logits = proto_only_logits(z, P, log_tau, cfg)
    ce = F.cross_entropy(logits, y, weight=class_weight, label_smoothing=0.05)
    align = 1.0 - torch.sum(z * P[y], dim=1).mean()
    reg_tau = (log_tau - math.log(cfg.tau_init)) ** 2
    return ce + cfg.lambda_align * align + cfg.lambda_tau * reg_tau


@torch.inference_mode()
def predict_fedproto_loader(
    encoder: nn.Module,
    loader: DataLoader,
    prototypes: torch.Tensor,
    log_tau: torch.Tensor,
    cfg: Config,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    encoder.eval()
    probs_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with torch.amp.autocast(autocast_device, enabled=amp_enabled):
            z = encoder(xb)
            logits = proto_only_logits(z, prototypes.to(device), log_tau.to(device), cfg)
            probs = torch.softmax(logits, dim=1)
        probs_all.append(probs.detach().cpu().numpy())
        y_all.append(yb.detach().cpu().numpy())
    return np.concatenate(probs_all, axis=0), np.concatenate(y_all, axis=0)


def evaluate_fedproto(encoder: nn.Module, loader: DataLoader, prototypes: torch.Tensor, log_tau: torch.Tensor, num_classes: int, cfg: Config, device: torch.device) -> Dict[str, float]:
    probs, labels = predict_fedproto_loader(encoder, loader, prototypes, log_tau, cfg, device)
    return summarize_probs(probs, labels, num_classes=num_classes, ece_bins=cfg.ece_bins)


@torch.inference_mode()
def extract_mean_prototypes(
    encoder: nn.Module,
    loader: DataLoader,
    num_classes: int,
    emb_dim: int,
    device: torch.device,
    cfg: Config,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    encoder.eval()
    sums = torch.zeros(num_classes, emb_dim, device=device)
    counts = torch.zeros(num_classes, device=device)
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        z = encoder(xb)
        sums.index_add_(0, yb, z)
        counts.index_add_(0, yb, torch.ones_like(yb, dtype=torch.float32, device=device))
    proto = torch.zeros_like(sums)
    present = counts >= max(1, cfg.min_proto_samples_per_class)
    proto[present] = F.normalize(sums[present] / counts[present].unsqueeze(1), dim=1)
    return proto.detach().cpu(), present.detach().cpu(), counts.detach().cpu()


def train_fedproto_client(
    global_encoder: nn.Module,
    train_loader: DataLoader,
    global_proto: torch.Tensor,
    log_tau_init: torch.Tensor,
    num_classes: int,
    cfg: Config,
    device: torch.device,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, Dict[str, torch.Tensor]]:
    local_encoder = clone_model(global_encoder).to(device)
    log_tau = nn.Parameter(log_tau_init.clone().to(device).float())
    labels = train_loader.dataset.y.detach().cpu().numpy()
    class_weight = make_class_weight_from_labels(labels, num_classes, device, cfg)
    optimizer = torch.optim.AdamW(list(local_encoder.parameters()) + [log_tau], lr=cfg.lr_encoder, weight_decay=cfg.weight_decay)
    autocast_device = "cuda" if device.type == "cuda" else "cpu"
    amp_enabled = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp_enabled)
    for _epoch in range(cfg.local_epochs):
        local_encoder.train()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(autocast_device, enabled=amp_enabled):
                z = local_encoder(xb)
                loss = fedproto_loss(z, yb, global_proto.to(device), log_tau, cfg, class_weight=class_weight)
            if amp_enabled:
                scaler.scale(loss).backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(list(local_encoder.parameters()) + [log_tau], cfg.grad_clip)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward()
                if cfg.grad_clip is not None and cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(list(local_encoder.parameters()) + [log_tau], cfg.grad_clip)
                optimizer.step()
    emp, present, counts = extract_mean_prototypes(local_encoder, train_loader, num_classes, cfg.emb_dim, device, cfg)
    state = {k: v.detach().cpu().clone() for k, v in local_encoder.state_dict().items()}
    return state, log_tau.detach().cpu().clone(), {"empirical": emp, "present": present, "counts": counts}


def aggregate_fedproto_prototypes(
    old_proto: torch.Tensor,
    proto_summaries: Dict[str, Dict[str, torch.Tensor]],
    cfg: Config,
    device: torch.device,
) -> torch.Tensor:
    old_proto = old_proto.to(device)
    num_classes, emb_dim = old_proto.shape
    sums = torch.zeros(num_classes, emb_dim, device=device)
    weights = torch.zeros(num_classes, device=device)
    for s in proto_summaries.values():
        emp = s["empirical"].to(device)
        present = s["present"].to(device).bool()
        counts = s["counts"].to(device).float()
        w = torch.where(present, torch.sqrt(torch.clamp(counts, min=0.0)), torch.zeros_like(counts))
        sums += w.unsqueeze(1) * emp
        weights += w
    new_proto = old_proto.clone()
    keep = weights > 0
    new_proto[keep] = sums[keep] / weights[keep].unsqueeze(1)
    new_proto = F.normalize(cfg.server_proto_momentum * old_proto + (1.0 - cfg.server_proto_momentum) * new_proto, dim=1)
    return new_proto.detach()


def adapt_fedproto_support_prototypes(
    encoder: nn.Module,
    support_loader: DataLoader,
    adapt_val_loader: DataLoader,
    global_proto: torch.Tensor,
    init_log_tau: torch.Tensor,
    num_classes: int,
    cfg: Config,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float], Dict[str, Any]]:
    encoder = clone_model(encoder).to(device)
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad = False
    global_proto = global_proto.to(device)
    baseline_metrics = evaluate_fedproto(encoder, adapt_val_loader, global_proto, init_log_tau.to(device), num_classes, cfg, device)
    baseline_score = personalization_selection_score(baseline_metrics)

    support_proto, present, _ = extract_mean_prototypes(encoder, support_loader, num_classes, cfg.emb_dim, device, cfg)
    support_proto = support_proto.to(device)
    present = present.to(device).bool()
    personal_proto = global_proto.clone()
    if present.any():
        personal_proto[present] = F.normalize(0.20 * global_proto[present] + 0.80 * support_proto[present], dim=1)
    log_tau = nn.Parameter(init_log_tau.clone().to(device).float())
    optimizer = torch.optim.AdamW([log_tau], lr=cfg.lr_personal * cfg.personal_tau_lr_mult, weight_decay=0.0)
    best_proto = personal_proto.detach().cpu().clone()
    best_log_tau = log_tau.detach().cpu().clone()
    best_metrics = evaluate_fedproto(encoder, adapt_val_loader, personal_proto, log_tau, num_classes, cfg, device)
    best_score = personalization_selection_score(best_metrics)
    no_improve = 0
    for _epoch in range(cfg.personalization_epochs):
        for xb, yb in support_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            z = encoder(xb)
            loss = fedproto_loss(z, yb, personal_proto, log_tau, cfg, class_weight=None)
            loss.backward()
            optimizer.step()
        m = evaluate_fedproto(encoder, adapt_val_loader, personal_proto, log_tau, num_classes, cfg, device)
        s = personalization_selection_score(m)
        if s > best_score:
            best_score = s
            best_metrics = m
            best_log_tau = log_tau.detach().cpu().clone()
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= cfg.personalization_patience:
            break

    use_personal = best_score > baseline_score + cfg.personalization_gate_score_margin
    gate = {
        "used_personalization": bool(use_personal),
        "baseline_metrics": baseline_metrics,
        "baseline_score": float(baseline_score),
        "personalized_metrics": best_metrics,
        "personalized_score": float(best_score),
    }
    if use_personal:
        return best_proto, best_log_tau, best_metrics, gate
    return global_proto.detach().cpu().clone(), init_log_tau.detach().cpu().clone(), baseline_metrics, gate


def train_fedproto_fold(loaders: Dict[str, object], cfg: Config, device: torch.device) -> Dict[str, Any]:
    num_classes = loaders["num_classes"]
    feature_dim = loaders["feature_dim"]
    heldout_id = loaders["heldout_id"]
    train_ids = loaders["train_ids"]
    train_clients = loaders["train_clients"]
    heldout = loaders["heldout"]

    global_encoder = TemporalEncoder(feature_dim, cfg).to(device)
    global_proto = F.normalize(torch.randn(num_classes, cfg.emb_dim, device=device), dim=1)
    client_log_tau = {sid: torch.log(torch.tensor(cfg.tau_init, dtype=torch.float32)) for sid in train_ids}
    best_snapshot: Optional[Dict[str, Any]] = None
    best_score = -1e9
    history: List[Dict[str, float]] = []

    round_bar = tqdm(range(1, cfg.rounds + 1), desc=f"FedProto {heldout_id}", leave=True)
    for rnd in round_bar:
        states: List[Dict[str, torch.Tensor]] = []
        weights: List[float] = []
        summaries: Dict[str, Dict[str, torch.Tensor]] = {}
        for sid in train_ids:
            state, log_tau, summary = train_fedproto_client(global_encoder, train_clients[sid]["train"], global_proto, client_log_tau[sid], num_classes, cfg, device)
            states.append(state)
            weights.append(train_clients[sid]["n_train"])
            summaries[sid] = summary
            client_log_tau[sid] = log_tau
        global_encoder.load_state_dict(average_state_dicts(states, weights))
        global_proto = aggregate_fedproto_prototypes(global_proto, summaries, cfg, device)

        train_size_weights = torch.tensor([train_clients[sid]["n_train"] for sid in train_ids], dtype=torch.float32, device=device)
        tau_stack = torch.stack([client_log_tau[sid].float().to(device) for sid in train_ids], dim=0)
        mean_log_tau = (train_size_weights * tau_stack).sum() / train_size_weights.sum()

        per_client_metrics: Dict[str, Dict[str, float]] = {}
        val_sizes: Dict[str, int] = {}
        for sid in train_ids:
            per_client_metrics[sid] = evaluate_fedproto(global_encoder, train_clients[sid]["val"], global_proto, mean_log_tau, num_classes, cfg, device)
            val_sizes[sid] = len(train_clients[sid]["val"].dataset)
        mean_f1, mean_acc, mean_ece, score = aggregate_round_metrics(per_client_metrics, val_sizes, cfg)
        history.append({"round": rnd, "mean_val_f1": mean_f1, "mean_val_acc": mean_acc, "mean_val_ece": mean_ece})
        round_bar.set_postfix({"val_f1": f"{mean_f1:.4f}", "val_acc": f"{mean_acc:.4f}", "val_ece": f"{mean_ece:.4f}"})
        if score > best_score:
            best_score = score
            best_snapshot = {
                "encoder_state": {k: v.detach().cpu().clone() for k, v in global_encoder.state_dict().items()},
                "global_proto": global_proto.detach().cpu().clone(),
                "mean_log_tau": mean_log_tau.detach().cpu().clone(),
                "history": copy.deepcopy(history),
            }

    assert best_snapshot is not None
    global_encoder.load_state_dict(best_snapshot["encoder_state"])
    global_proto = best_snapshot["global_proto"].to(device)
    mean_log_tau = best_snapshot["mean_log_tau"].to(device)
    global_test = evaluate_fedproto(global_encoder, heldout["test"], global_proto, mean_log_tau, num_classes, cfg, device)
    personal_proto, personal_log_tau, adapt_metrics, gate = adapt_fedproto_support_prototypes(
        global_encoder, heldout["support"], heldout["adapt_val"], global_proto, mean_log_tau, num_classes, cfg, device
    )
    personal_test = evaluate_fedproto(global_encoder, heldout["test"], personal_proto.to(device), personal_log_tau.to(device), num_classes, cfg, device)
    return {
        "heldout_id": heldout_id,
        "method": "FedProto",
        "variant": "Global+Personalized",
        "global_test": global_test,
        "personalized_test": personal_test,
        "adapt_val": adapt_metrics,
        "adapt_gate": gate,
        "history": best_snapshot["history"],
    }


def maybe_cache_dataset_locally(cfg: Config) -> Config:
    """Copy UCI HAR from Google Drive to /content to avoid Colab Drive transport-endpoint errors."""
    if getattr(cfg, "dataset_name", "pamap2").lower() != "uci_har":
        return cfg
    if not bool(getattr(cfg, "cache_drive_dataset_locally", True)):
        return cfg
    src_root = find_uci_har_root(cfg.data_dir)
    cache_root = Path(getattr(cfg, "local_data_cache_dir", "/content/uci_har_cached")) / "UCI HAR Dataset"
    required = cache_root / "train" / "Inertial Signals" / "body_acc_x_train.txt"
    if not required.exists():
        print(f"[UCI HAR] Copying dataset from Drive to local cache: {cache_root}")
        if cache_root.exists():
            shutil.rmtree(cache_root)
        cache_root.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_root, cache_root)
    else:
        print(f"[UCI HAR] Using local cached dataset: {cache_root}")
    cfg_local = copy.copy(cfg)
    cfg_local.data_dir = str(cache_root)
    return cfg_local


def method_metric_rows_from_result(dataset: str, heldout_id: str, result: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    def add(method: str, variant: str, metrics: Dict[str, float], used_personalization: Optional[bool] = None) -> None:
        rows.append({
            "dataset": dataset,
            "subject": heldout_id,
            "method": method,
            "variant": variant,
            "accuracy": float(metrics["acc"]),
            "macro_f1": float(metrics["f1"]),
            "ece": float(metrics["ece"]),
            "brier": float(metrics["brier"]),
            "used_personalization": used_personalization,
        })

    method = result.get("method", "")
    if method == "CDPL":
        add("CDPL", "Global", result["global_test"], False)
        add("CDPL", "Personalized", result["personalized_test"], bool(result.get("adapt_gate", {}).get("used_personalization", False)))
    elif method == "FedProto":
        add("FedProto", "Global", result["global_test"], False)
        add("FedProto", "Personalized", result["personalized_test"], bool(result.get("adapt_gate", {}).get("used_personalization", False)))
    else:
        add(result["method"], result["variant"], result["test"], None)
    return rows


def save_all_method_results_tables(cfg: Config, rows: List[Dict[str, Any]]) -> None:
    out_dir = Path(cfg.save_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    fold_df = pd.DataFrame(rows)
    fold_csv = out_dir / f"{cfg.dataset_name}_all_methods_fold_results.csv"
    fold_df.to_csv(fold_csv, index=False)

    metric_cols = ["accuracy", "macro_f1", "ece", "brier"]
    summary_rows: List[Dict[str, Any]] = []
    wide_rows: List[Dict[str, Any]] = []
    for (dataset, method, variant), g in fold_df.groupby(["dataset", "method", "variant"], dropna=False):
        wide = {"dataset": dataset, "method": method, "variant": variant, "n_subjects": int(g["subject"].nunique())}
        for metric in metric_cols:
            stats_dict = metric_mean_std_ci(g[metric].tolist())
            summary_rows.append({
                "dataset": dataset,
                "method": method,
                "variant": variant,
                "metric": metric,
                **stats_dict,
            })
            wide[metric] = f"{stats_dict['mean']:.4f} ± {stats_dict['std']:.4f} [{stats_dict['ci95_low']:.4f}, {stats_dict['ci95_high']:.4f}]"
        wide_rows.append(wide)

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / f"{cfg.dataset_name}_all_methods_summary_ci.csv", index=False)
    wide_df = pd.DataFrame(wide_rows).sort_values(["method", "variant"])
    wide_df.to_csv(out_dir / f"{cfg.dataset_name}_all_methods_paper_table_all_variants.csv", index=False)

    # Reviewer-table subset: the standard variants usually reported in the paper.
    selected = [
        ("FedAvg", "Global", "Global baseline"),
        ("FedProto", "Personalized", "Prototype baseline"),
        ("FedRep", "Personalized", "Strong personalized baseline"),
        ("CDPL", "Personalized", "Structured prototype deformation"),
    ]
    selected_rows: List[Dict[str, Any]] = []
    for method, variant, observation in selected:
        match = wide_df[(wide_df["method"] == method) & (wide_df["variant"] == variant)]
        if len(match) == 0:
            continue
        row = match.iloc[0].to_dict()
        row["main_observation"] = observation
        selected_rows.append(row)
    pd.DataFrame(selected_rows).to_csv(out_dir / f"{cfg.dataset_name}_review2_paper_table.csv", index=False)
    print(f"[RESULTS] Saved fold results to: {fold_csv}")
    print(f"[RESULTS] Saved summary CI table to: {out_dir / f'{cfg.dataset_name}_all_methods_summary_ci.csv'}")
    print(f"[RESULTS] Saved reviewer paper table to: {out_dir / f'{cfg.dataset_name}_review2_paper_table.csv'}")


def run_loso_all_methods(cfg: Config) -> Dict[str, Any]:
    """Run Review-2 UCI HAR LOSO with FedAvg, FedProto, FedRep, and CDPL, then save combined CSV files."""
    set_seed(cfg.seed)
    device = torch.device(cfg.device)
    cfg_load = maybe_cache_dataset_locally(cfg)
    subject_arrays, label_map = load_dataset_subject_arrays(cfg_load)

    subject_ids = sorted(subject_arrays.keys())
    if cfg.run_single_heldout is not None:
        if cfg.run_single_heldout not in subject_ids:
            raise ValueError(f"Unknown held-out subject {cfg.run_single_heldout!r}. Available: {subject_ids}")
        subject_ids = [cfg.run_single_heldout]

    methods = tuple(m.lower() for m in getattr(cfg, "methods_to_run", ("fedavg", "fedproto", "fedrep", "cdpl")))
    valid = {"fedavg", "fedproto", "fedrep", "cdpl"}
    unknown = [m for m in methods if m not in valid]
    if unknown:
        raise ValueError(f"Unknown methods_to_run entries: {unknown}. Valid: {sorted(valid)}")

    print("\n================ REVIEW-2 MULTI-METHOD CONFIG ================")
    for k, v in asdict(cfg).items():
        print(f"{k}: {v}")
    print("\n================ SUBJECTS ================")
    print(subject_ids)
    print(f"Label map (official activity_id -> class_idx): {label_map}")
    if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har":
        print(f"Activity names: {UCI_HAR_ACTIVITY_NAMES}")
        print("Input tensor per example: [128 time steps, 9 inertial channels]")

    all_rows: List[Dict[str, Any]] = []
    detailed_results: List[Dict[str, Any]] = []

    for fold_idx, heldout_id in enumerate(subject_ids):
        print(f"\n{'='*18} HELD-OUT {heldout_id} {'='*18}")
        fold = build_fold_data(subject_arrays, heldout_id, label_map, cfg)
        loaders = fold_to_loaders(fold, cfg)
        print("Fold summary:")
        for sid, stats_dict in loaders["summary"].items():
            stats_str = ", ".join([f"{k}={v}" for k, v in stats_dict.items()])
            print(f"  {sid}: {stats_str}")

        method_results: List[Dict[str, Any]] = []
        for method_idx, method in enumerate(methods):
            set_seed(cfg.seed + 1000 * fold_idx + 37 * method_idx)
            print(f"\n--- Running {method.upper()} for held-out {heldout_id} ---")
            if method == "fedavg":
                result = train_fedavg_fold(loaders, cfg, device)
            elif method == "fedproto":
                result = train_fedproto_fold(loaders, cfg, device)
            elif method == "fedrep":
                result = train_fedrep_fold(loaders, cfg, device)
            elif method == "cdpl":
                cdpl_result = train_one_fold(loaders, cfg, device)
                try:
                    saved_dir = save_fold_artifacts(cdpl_result, cfg)
                    print(f"Saved CDPL inference artifacts to: {saved_dir}")
                except Exception as exc:
                    print(f"[WARN] Could not save CDPL artifacts for {heldout_id}: {exc}")
                result = {
                    **cdpl_result,
                    "method": "CDPL",
                    "variant": "Global+Personalized",
                }
            else:
                raise AssertionError(method)

            method_results.append(result)
            new_rows = method_metric_rows_from_result(cfg.dataset_name, heldout_id, result)
            all_rows.extend(new_rows)
            save_all_method_results_tables(cfg, all_rows)  # incremental save after every method
            for row in new_rows:
                print(
                    f"{row['method']} {row['variant']} -> "
                    f"ACC: {row['accuracy']:.4f}, Macro-F1: {row['macro_f1']:.4f}, "
                    f"ECE: {row['ece']:.4f}, Brier: {row['brier']:.4f}"
                )

        detailed_results.append({"heldout_id": heldout_id, "method_results": method_results})

    save_all_method_results_tables(cfg, all_rows)
    save_json(Path(cfg.save_dir) / f"{cfg.dataset_name}_all_methods_detailed_results.json", {
        "config": asdict(cfg),
        "label_map": {int(k): int(v) for k, v in label_map.items()},
        "idx_to_activity": inverse_label_map(label_map),
        "activity_names": UCI_HAR_ACTIVITY_NAMES if getattr(cfg, "dataset_name", "pamap2").lower() == "uci_har" else None,
        "fold_rows": all_rows,
    })
    return {"rows": all_rows, "label_map": label_map, "folds": detailed_results}

## Main (2)

Manual execution launcher. In this notebook it is disabled by default using RUN_FULL_BENCHMARK = False so Run All does not accidentally start a full LOSO benchmark.

> Safety note: this cell will not start training unless `RUN_FULL_BENCHMARK` is set to `True`.

In [ ]:
# ============================================================
# Main
# ============================================================

RUN_FULL_BENCHMARK = False  # Set True only when dataset paths are correct and you want to start training.
if RUN_FULL_BENCHMARK:
    # UCI HAR Review-2 benchmark.
    # First run one subject for debugging. After it succeeds, set run_single_heldout=None for full 30-subject LOSO.
    cfg = Config(
        dataset_name="uci_har",
        data_dir="/kaggle/input/datasets/duanepm/uci-har/UCI HAR Dataset",
        save_dir="/kaggle/working/cdpl_uci_har_runs",
        seq_len=128,
        stride=64,
        purge_gap_raw=0,
        uci_purge_gap_windows=1,
        min_windows_per_partition=4,
        rounds=15,
        local_epochs=5,
        personalization_epochs=10,
        lr_personal=7.5e-3,
        methods_to_run=("fedavg", "fedproto", "fedrep", "cdpl"),
        cache_drive_dataset_locally=False,
        local_data_cache_dir="/kaggle/working/uci_har_cached",
        run_single_heldout=None,  # change to None for full 30-subject LOSO
    )
    results = run_loso_all_methods(cfg)